# One decode step of grouped-query attention with a KV cache

During decoding, each sequence in the batch adds exactly one token. Its key and value are written into a preallocated cache, and its query attends over everything cached so far, including itself. Sequences in a batch have different lengths, so each one writes at its own position and must ignore the unused tail of its cache buffer.

Write `decode_step(q, k_new, v_new, k_cache, v_cache, lengths)` and return the tuple `(out, k_cache, v_cache, new_lengths)`.

## Inputs (all float64 except `lengths`)
- `q`: `(B, H, D)`, the new token's query for each of `H` query heads.
- `k_new`, `v_new`: `(B, Hkv, D)`, the new token's key and value for each of `Hkv` KV heads. `H` is a multiple of `Hkv`; query head `h` uses KV head `h // (H // Hkv)` (grouped-query attention: consecutive query heads share a KV head).
- `k_cache`, `v_cache`: `(B, Hkv, S, D)` buffers. For sequence `b`, positions `0 .. lengths[b] - 1` hold valid entries; everything after is garbage (arbitrary finite values) and must not affect the result.
- `lengths`: int64 `(B,)` with `0 <= lengths[b] < S`.

## What to compute
1. Write `k_new[b]` and `v_new[b]` into position `lengths[b]` of sequence `b`'s cache (for every KV head).
2. For each `b` and query head `h`, attend over positions `0 .. lengths[b]` inclusive:
   $$\text{out}[b,h] = \sum_{j \le \ell_b} \operatorname{softmax}_j\!\Big(\tfrac{q_{b,h} \cdot k_{b,g(h),j}}{\sqrt D}\Big)\, v_{b,g(h),j}$$
3. `new_lengths = lengths + 1` (int64).

## Output
`out` `(B, H, D)` float64, the two updated caches `(B, Hkv, S, D)` (garbage positions other than the newly written one are returned unchanged), and `new_lengths`. You may update the caches in place or on a copy. No Python loops over the batch or heads.

This problem runs in **Colab** with a CPU runtime (PyTorch isn't available in the browser).

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def decode_step(q, k_new, v_new, k_cache, v_cache, lengths):
    """Return (out (B, H, D), k_cache, v_cache, new_lengths)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[2.0, 2.0]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0], [9.0, 9.0], [9.0, 9.0]]]], "dtype": "float64"}, {"__nd__": [[[[4.0, 0.0], [9.0, 9.0], [9.0, 9.0]]]], "dtype": "float64"}, {"__nd__": [1], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[2.6604769013466862, 1.3395230986533138]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0], [1.0, 0.0], [9.0, 9.0]]]], "dtype": "float64"}, {"__nd__": [[[[4.0, 0.0], [2.0, 2.0], [9.0, 9.0]]]], "dtype": "float64"}, {"__nd__": [2], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[0.0, 1.0], [1.0, 0.0]], [[1.0, 1.0], [0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[1.0, 1.0]], [[1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[1.0, -1.0]], [[3.0, 3.0]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [0, 2], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[1.0, -1.0], [1.0, -1.0]], [[2.018525395127931, 2.018525395127931], [1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, -1.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [3.0, 3.0]]]], "dtype": "float64"}, {"__nd__": [1, 3], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[-0.6402624053903738, 0.7323017147112972, -1.170530808407683, -1.434281459674122], [0.6398520751723783, 0.7543689046395509, -0.958933707794222, 0.5623976772929592], [-0.2916324304147521, 0.3012921765535601, -1.2609602800655342, 0.8328944493608352], [1.203258954116498, 0.6370732356261833, 0.5583399616951433, -3.772275156122734]], [[0.2606297490669398, -0.025445316692543535, -0.1470455068463331, -0.63057798997102], [0.0553649749135539, 0.41211740512833367, -0.263788160540625, -0.46332413970987085], [1.2297538109392712, -1.1053671072691253, 1.0301551978238768, 0.17681246373052467], [-0.8043056452824273, -0.28998189606931385, -0.9199936028121969, 0.6750645184094394]], [[0.3479017619704715, -0.5567961751531812, -1.102218299054223, 0.30171609350107054], [0.9573856068419316, -0.11383634938188146, 0.41835250088922316, -0.37602715054974273], [0.06756920600942723, -0.2912829113936885, 0.2940421429749802, -1.509772513035727], [0.6441905231896892, -0.2299324816033148, 0.3586080676493852, -0.340310127466209]]], "dtype": "float64"}, {"__nd__": [[[0.3203611893741035, -1.0725628041289215, 1.189154412488462, -1.7035445876372979], [-1.0392388147791862, 0.23566606953963426, 1.462842285184577, 0.2781263018080308]], [[-0.24790845527908614, -1.4250901432432643, -0.19127666443013225, -0.019877011042093756], [1.690568645339397, 0.6221252216057821, -1.5290928749284465, 2.0267790952431928]], [[-0.39500987549879313, -0.8794597147043167, 1.4748226520869099, -0.049755760296968106], [-0.3674025993780988, 0.2187859889352313, 0.8448887803757161, 0.9933362044496503]]], "dtype": "float64"}, {"__nd__": [[[-1.3752024000527405, 1.9984814702717295, 0.9468615879956256, -0.37920106201315124], [-0.8186598515214158, -0.9690124307582063, 0.12337843757618257, -0.6480162941896269]], [[-0.7648739652865205, 0.8112544049416356, 0.36456734382632344, -0.3945712584237112], [0.7342475961853729, 1.3673786263056709, -1.0944963126296459, -0.6032918841422477]], [[0.9426218402198768, 0.7189405658150543, 0.22669885643799229, 1.1624211875584574], [-1.0882117291190974, -1.479137833771104, -0.8665073572768981, 0.12254409553604008]]], "dtype": "float64"}, {"__nd__": [[[[40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]], [[40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]]], [[[1.801634869866125, 1.31510376473437, 0.357380410658956, -1.2083186322821715], [-0.004454133120083229, 0.6564749350763358, -1.2883614637495544, 0.39512206018200824], [0.42986369482223, 0.6960427239628685, -1.184117966757189, -0.6617025720390349], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]], [[1.0039615758421696, -0.6179070447076008, 1.8220113633283233, -1.3204309700132935], [-0.6615280218152191, 0.9350499881140221, 0.049054613825311656, 2.002392583645255], [0.18851919251246557, -0.6331940901922267, -0.37756350523280824, -1.0911461176191954], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]]], [[[0.16100957671534466, -0.5855288241233366, -1.341219714076669, -1.401520214917428], [0.5026828498748657, 0.989713033285805, -0.1642945926252907, -1.0743648582284346], [0.8730421526217066, -1.2803939447145731, -0.7130680950592722, 0.6210178535400985], [-2.2501411735745918, 0.38636959756630584, -0.5816408364095031, 0.10927969747781388], [-0.07570152622082311, 0.20211439504395987, 0.6941719367070082, -0.7583697508984092], [40.0, 40.0, 40.0, 40.0]], [[0.7875882217058694, 0.844078680578592, 0.07559361074288512, -1.4267738509897323], [-0.13504510003701392, -0.7695146401767057, -1.4227417685154136, 0.25845279091298756], [-0.5685494541476426, -1.0298044380114637, -1.0430010800715654, 0.26841707970891465], [0.35867194917034445, 1.3224574697668332, -0.013914668524093734, 1.0418397592128221], [1.4022648267725224, 1.1501656361496921, -2.3653039062769743, 1.228683719203421], [40.0, 40.0, 40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]], [[-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]]], [[[1.0868307847683634, -0.050604063111342405, -0.2831250656795347, 1.643251614242697], [-1.2826492440738984, -0.5856577998413593, -0.47258767675848407, 0.5863372815313004], [-0.663535198304047, -0.6134178486140281, -1.6051493968851136, 0.7293494040178566], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]], [[-0.0072113596587564995, -0.5611981104091591, -0.8676167643217026, 3.0660367390488967], [-0.07734505972421603, -2.016660690233245, -0.6486006079033346, 0.6780397271239971], [-0.5000084331847482, 1.3604462024852575, 1.0023982728439578, -0.1523386358242358], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]]], [[[1.2969153998005238, -0.345672529464465, 0.8545842348534083, -0.4889690638420449], [1.760667296993123, 0.19921798301385701, -0.3820022921434805, 2.552424025371081], [-0.3244718562854392, -1.2212233497261362, 0.2019100019601099, -0.03883503855807973], [1.066324553166257, -0.9216339244978112, 0.8047169314794976, 0.8527484705742913], [-0.6676872919350705, 0.16324400572267767, -0.8307519568543374, 2.3458080738406677], [-30.0, -30.0, -30.0, -30.0]], [[-0.005876026696904225, 0.7677891353446865, -0.6104866460656994, -0.1857739586715857], [-1.416489366414042, -0.8274022267661552, 2.755807558275951, 1.0412431916947957], [-0.7814240416773149, -1.3373972415282913, -0.9755828305996581, -0.02169085951373511], [0.034727788359190326, -0.7443606009308561, -1.2865744667376442, 1.4223785052484463], [0.4516854512950959, -0.3745680195679353, -0.22066120781608659, -0.5295304591256691], [-30.0, -30.0, -30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [0, 3, 5], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[-1.3752024000527405, 1.9984814702717295, 0.9468615879956256, -0.37920106201315124], [-1.3752024000527405, 1.9984814702717295, 0.9468615879956256, -0.37920106201315124], [-0.8186598515214158, -0.9690124307582063, 0.12337843757618257, -0.6480162941896269], [-0.8186598515214158, -0.9690124307582063, 0.12337843757618257, -0.6480162941896269]], [[-0.19635253410548695, -0.13613434134612853, -0.5536457523728645, 0.7977018862892318], [-0.25073864839030857, -0.21953528768360048, -0.6290837598851131, 0.8335021673672975], [0.014387293487756402, -0.13762309635000483, -0.627844668662637, 2.0319442264509138], [0.15196011432632456, -0.16846706196983016, -0.548282389131692, 0.16900898662614386]], [[0.5574157015646599, -0.5584649024981373, 0.3127910570529448, 0.4721543774044569], [0.6291989803561168, -0.12630846942010246, 0.014615560114132153, 1.17377316410419], [-0.3950887981162029, -0.13851539098412347, -0.25349966461651846, 0.13520114231603322], [-0.4165036694245545, -0.4812801613210284, -0.26675018542422496, 0.25361297947643824]]], "dtype": "float64"}, {"__nd__": [[[[0.3203611893741035, -1.0725628041289215, 1.189154412488462, -1.7035445876372979], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]], [[-1.0392388147791862, 0.23566606953963426, 1.462842285184577, 0.2781263018080308], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]]], [[[1.801634869866125, 1.31510376473437, 0.357380410658956, -1.2083186322821715], [-0.004454133120083229, 0.6564749350763358, -1.2883614637495544, 0.39512206018200824], [0.42986369482223, 0.6960427239628685, -1.184117966757189, -0.6617025720390349], [-0.24790845527908614, -1.4250901432432643, -0.19127666443013225, -0.019877011042093756], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]], [[1.0039615758421696, -0.6179070447076008, 1.8220113633283233, -1.3204309700132935], [-0.6615280218152191, 0.9350499881140221, 0.049054613825311656, 2.002392583645255], [0.18851919251246557, -0.6331940901922267, -0.37756350523280824, -1.0911461176191954], [1.690568645339397, 0.6221252216057821, -1.5290928749284465, 2.0267790952431928], [40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0]]], [[[0.16100957671534466, -0.5855288241233366, -1.341219714076669, -1.401520214917428], [0.5026828498748657, 0.989713033285805, -0.1642945926252907, -1.0743648582284346], [0.8730421526217066, -1.2803939447145731, -0.7130680950592722, 0.6210178535400985], [-2.2501411735745918, 0.38636959756630584, -0.5816408364095031, 0.10927969747781388], [-0.07570152622082311, 0.20211439504395987, 0.6941719367070082, -0.7583697508984092], [-0.39500987549879313, -0.8794597147043167, 1.4748226520869099, -0.049755760296968106]], [[0.7875882217058694, 0.844078680578592, 0.07559361074288512, -1.4267738509897323], [-0.13504510003701392, -0.7695146401767057, -1.4227417685154136, 0.25845279091298756], [-0.5685494541476426, -1.0298044380114637, -1.0430010800715654, 0.26841707970891465], [0.35867194917034445, 1.3224574697668332, -0.013914668524093734, 1.0418397592128221], [1.4022648267725224, 1.1501656361496921, -2.3653039062769743, 1.228683719203421], [-0.3674025993780988, 0.2187859889352313, 0.8448887803757161, 0.9933362044496503]]]], "dtype": "float64"}, {"__nd__": [[[[-1.3752024000527405, 1.9984814702717295, 0.9468615879956256, -0.37920106201315124], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]], [[-0.8186598515214158, -0.9690124307582063, 0.12337843757618257, -0.6480162941896269], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]]], [[[1.0868307847683634, -0.050604063111342405, -0.2831250656795347, 1.643251614242697], [-1.2826492440738984, -0.5856577998413593, -0.47258767675848407, 0.5863372815313004], [-0.663535198304047, -0.6134178486140281, -1.6051493968851136, 0.7293494040178566], [-0.7648739652865205, 0.8112544049416356, 0.36456734382632344, -0.3945712584237112], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]], [[-0.0072113596587564995, -0.5611981104091591, -0.8676167643217026, 3.0660367390488967], [-0.07734505972421603, -2.016660690233245, -0.6486006079033346, 0.6780397271239971], [-0.5000084331847482, 1.3604462024852575, 1.0023982728439578, -0.1523386358242358], [0.7342475961853729, 1.3673786263056709, -1.0944963126296459, -0.6032918841422477], [-30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0]]], [[[1.2969153998005238, -0.345672529464465, 0.8545842348534083, -0.4889690638420449], [1.760667296993123, 0.19921798301385701, -0.3820022921434805, 2.552424025371081], [-0.3244718562854392, -1.2212233497261362, 0.2019100019601099, -0.03883503855807973], [1.066324553166257, -0.9216339244978112, 0.8047169314794976, 0.8527484705742913], [-0.6676872919350705, 0.16324400572267767, -0.8307519568543374, 2.3458080738406677], [0.9426218402198768, 0.7189405658150543, 0.22669885643799229, 1.1624211875584574]], [[-0.005876026696904225, 0.7677891353446865, -0.6104866460656994, -0.1857739586715857], [-1.416489366414042, -0.8274022267661552, 2.755807558275951, 1.0412431916947957], [-0.7814240416773149, -1.3373972415282913, -0.9755828305996581, -0.02169085951373511], [0.034727788359190326, -0.7443606009308561, -1.2865744667376442, 1.4223785052484463], [0.4516854512950959, -0.3745680195679353, -0.22066120781608659, -0.5295304591256691], [-1.0882117291190974, -1.479137833771104, -0.8665073572768981, 0.12254409553604008]]]], "dtype": "float64"}, {"__nd__": [1, 4, 6], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[0.09573071096425678, -0.17847043139921506, 0.20262400099114752], [-1.6057480583244386, 1.8122301162723733, -0.602658614543728], [-1.539659308496411, 0.6188421885671495, -0.3548041301011767], [0.32485848577290377, -0.33960843062503854, -0.059740360479920165]], [[0.24577284373863384, -0.7466528839828983, 0.6787395958595579], [-0.46990009907954344, -0.8696871441704723, 0.07703242182250279], [0.44504127849104197, -0.2290793416186396, -0.8625197870795628], [0.6197855663086329, -1.7603287921227768, -1.0308641360355328]]], "dtype": "float64"}, {"__nd__": [[[0.03952289053338441, -1.3610593983050674, 0.027994264249169242], [-0.05486311801846381, 0.8987397888581683, -0.9147903518132915], [-0.6259065236416427, 0.3331816847010001, -2.4575635902058073], [3.1000422989145844, -0.698650730461769, -0.7298350527255578]], [[0.8611275109037129, -0.03983184143568413, -1.779428618703591], [0.6269273800926122, 0.8553778339992086, -0.4499462734759413], [-0.2816003582920249, 0.48598459725939197, -0.9087802499144495], [0.4383885642069085, 0.199298510444498, -0.674932615775918]]], "dtype": "float64"}, {"__nd__": [[[-1.3921018738118445, -0.22560583076108617, -0.8754222582601685], [1.0014102256801642, 0.14408536849992318, 0.7820845225598966], [0.13462193534445818, 0.26290111708503067, -0.7829989172303806], [0.6680474265721447, 1.7846982743070243, -0.3096875555175417]], [[-0.592774527714149, -0.15783670219035234, -0.48128028360112374], [-0.7014792986535402, 0.13819364396816675, -0.29091753305009044], [1.4388735938426587, 0.00020164295512421895, 0.3239119776314283], [0.9520218679998016, -0.30075585250454295, 1.4367365400889385]]], "dtype": "float64"}, {"__nd__": [[[[0.345584192064786, 0.8216181435011584, 0.33043707618338714], [-1.303157231604361, 0.9053558666731177, 0.4463745723640113], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]], [[0.5988462126346276, 0.03972210748165899, -0.2924567509650886], [-0.7819084623568421, -0.2571922406188707, 0.008142180518343508], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]], [[2.1178387550510482, -1.1120207626922813, -0.37760500712699807], [2.0427716074923303, 0.6467029962018469, 0.6630633723762617], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]], [[0.09548302746945433, 0.03558623705548571, -0.5062916583143148], [0.5937480717858228, 0.8911669542823284, 0.3208483045665637], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]]], [[[0.05410227877154389, 0.27279133916445375, -0.9821881249409777], [-1.107373047165193, 0.19958453284708083, -0.46674961687980204], [0.23550561173022522, 0.7595195224783792, -1.6487873663509485], [0.2543881165176173, 1.2246469675357323, -0.2975268443704732], [40.0, 40.0, 40.0]], [[0.8958830707775604, -0.3452157100512797, -1.4818182737222112], [-0.11001076471125099, -0.4458281530112322, 0.7753238220475741], [0.1936328483771538, -1.6308492324351012, -1.1951630801031998], [0.8837890365872553, 0.6797650174178466, -0.6402433659084887], [40.0, 40.0, 40.0]], [[0.8762421961143501, 0.256485627221562, -0.09482833896849817], [-0.25884806478784556, 1.0557428005332512, -2.2508542750785376], [-0.13865532509133732, 0.03300010398406011, -1.4253489608701877], [0.33281361313804664, -0.651281012443394, 0.8624447963157468], [40.0, 40.0, 40.0]], [[0.3829295827134724, -0.8757211434228455, -1.5143186317046384], [1.7533841175163727, -0.11129219318751944, -0.6885649476343216], [0.14425708806082496, -0.19141133048264922, 0.8521422642126877], [0.03392818243710029, 0.013749583618419497, -0.7145797210329641], [40.0, 40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.5239375129901318, -1.5246860380991518, -2.466229231351318], [0.6168787551543194, 2.547897815483126, -1.0009248488718936], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]], [[-0.8446711036516033, -0.31982625774322104, -0.9503996651679186], [0.006514985869748843, -1.1238662275656366, -1.092894369295243], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]], [[0.8333425966696618, -0.590434943289043, -1.0560789503141859], [-0.9004750699990517, -0.3905453443553229, 1.6273002546230673], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]], [[0.6636575710174066, 1.9724738539297917, 0.20916747233627678], [-0.5924100997529326, -0.12597918998664273, -0.07249854561981205], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]]], [[[0.4930281494994427, 0.16115931384552032, -0.9322203053719521], [2.8715673378134987, 0.8802586206615082, -1.1392946703429758], [-0.7796379162397445, 0.08697924857190435, -1.5547311319959862], [0.16863040701051427, -0.4590715557127591, 1.2262706003162174], [-30.0, -30.0, -30.0]], [[-1.6174674995236882, 1.109637999248523, 0.16810586912782435], [0.5484054521869972, -1.065124728803299, 1.8284302379955002], [2.020073367150445, -1.0647710426262125, 0.37281512186631627], [-0.6733024283901868, -0.0235699368607321, -1.2656369792245639], [-30.0, -30.0, -30.0]], [[0.5014829311105223, -0.6475606784161285, -0.23931242973230216], [-0.5636398464269645, -0.13346075483629405, -1.1705426351003028], [-0.43798807560230063, -0.20689292471224427, -0.33372600380413486], [0.05668995489379499, -0.2931022193567212, 0.7532114084393808], [-30.0, -30.0, -30.0]], [[-0.526514840115237, -1.2644927908197123, 0.5187849208779641], [-1.142518017652492, -0.7458563981883249, 0.35924465201120453], [0.40257336594453236, -0.400114751005071, -2.0192658104888017], [0.42051328729557524, 0.2595634588920596, -1.4123812154717754], [-30.0, -30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [2, 4], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[0.1402218564456504, 0.1753784245298193, -1.424696734101284], [0.5320592577343362, -0.2434320802014296, 0.09233760328797383], [0.10411124027641705, 0.2032284561921589, -0.669683070116322], [0.3787873662545685, 1.3954821914854225, -0.12400510485142692]], [[0.5862512260068968, 0.1324658086497529, -0.551558036431594], [0.5061813856609076, -0.4809263395282127, 0.46948761744623674], [0.049737089279343775, -0.22706423839632398, -0.3626025040162881], [-0.3574541067826204, -0.8279556733821983, 0.22102520688568383]]], "dtype": "float64"}, {"__nd__": [[[[0.345584192064786, 0.8216181435011584, 0.33043707618338714], [-1.303157231604361, 0.9053558666731177, 0.4463745723640113], [0.03952289053338441, -1.3610593983050674, 0.027994264249169242], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]], [[0.5988462126346276, 0.03972210748165899, -0.2924567509650886], [-0.7819084623568421, -0.2571922406188707, 0.008142180518343508], [-0.05486311801846381, 0.8987397888581683, -0.9147903518132915], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]], [[2.1178387550510482, -1.1120207626922813, -0.37760500712699807], [2.0427716074923303, 0.6467029962018469, 0.6630633723762617], [-0.6259065236416427, 0.3331816847010001, -2.4575635902058073], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]], [[0.09548302746945433, 0.03558623705548571, -0.5062916583143148], [0.5937480717858228, 0.8911669542823284, 0.3208483045665637], [3.1000422989145844, -0.698650730461769, -0.7298350527255578], [40.0, 40.0, 40.0], [40.0, 40.0, 40.0]]], [[[0.05410227877154389, 0.27279133916445375, -0.9821881249409777], [-1.107373047165193, 0.19958453284708083, -0.46674961687980204], [0.23550561173022522, 0.7595195224783792, -1.6487873663509485], [0.2543881165176173, 1.2246469675357323, -0.2975268443704732], [0.8611275109037129, -0.03983184143568413, -1.779428618703591]], [[0.8958830707775604, -0.3452157100512797, -1.4818182737222112], [-0.11001076471125099, -0.4458281530112322, 0.7753238220475741], [0.1936328483771538, -1.6308492324351012, -1.1951630801031998], [0.8837890365872553, 0.6797650174178466, -0.6402433659084887], [0.6269273800926122, 0.8553778339992086, -0.4499462734759413]], [[0.8762421961143501, 0.256485627221562, -0.09482833896849817], [-0.25884806478784556, 1.0557428005332512, -2.2508542750785376], [-0.13865532509133732, 0.03300010398406011, -1.4253489608701877], [0.33281361313804664, -0.651281012443394, 0.8624447963157468], [-0.2816003582920249, 0.48598459725939197, -0.9087802499144495]], [[0.3829295827134724, -0.8757211434228455, -1.5143186317046384], [1.7533841175163727, -0.11129219318751944, -0.6885649476343216], [0.14425708806082496, -0.19141133048264922, 0.8521422642126877], [0.03392818243710029, 0.013749583618419497, -0.7145797210329641], [0.4383885642069085, 0.199298510444498, -0.674932615775918]]]], "dtype": "float64"}, {"__nd__": [[[[1.5239375129901318, -1.5246860380991518, -2.466229231351318], [0.6168787551543194, 2.547897815483126, -1.0009248488718936], [-1.3921018738118445, -0.22560583076108617, -0.8754222582601685], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]], [[-0.8446711036516033, -0.31982625774322104, -0.9503996651679186], [0.006514985869748843, -1.1238662275656366, -1.092894369295243], [1.0014102256801642, 0.14408536849992318, 0.7820845225598966], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]], [[0.8333425966696618, -0.590434943289043, -1.0560789503141859], [-0.9004750699990517, -0.3905453443553229, 1.6273002546230673], [0.13462193534445818, 0.26290111708503067, -0.7829989172303806], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]], [[0.6636575710174066, 1.9724738539297917, 0.20916747233627678], [-0.5924100997529326, -0.12597918998664273, -0.07249854561981205], [0.6680474265721447, 1.7846982743070243, -0.3096875555175417], [-30.0, -30.0, -30.0], [-30.0, -30.0, -30.0]]], [[[0.4930281494994427, 0.16115931384552032, -0.9322203053719521], [2.8715673378134987, 0.8802586206615082, -1.1392946703429758], [-0.7796379162397445, 0.08697924857190435, -1.5547311319959862], [0.16863040701051427, -0.4590715557127591, 1.2262706003162174], [-0.592774527714149, -0.15783670219035234, -0.48128028360112374]], [[-1.6174674995236882, 1.109637999248523, 0.16810586912782435], [0.5484054521869972, -1.065124728803299, 1.8284302379955002], [2.020073367150445, -1.0647710426262125, 0.37281512186631627], [-0.6733024283901868, -0.0235699368607321, -1.2656369792245639], [-0.7014792986535402, 0.13819364396816675, -0.29091753305009044]], [[0.5014829311105223, -0.6475606784161285, -0.23931242973230216], [-0.5636398464269645, -0.13346075483629405, -1.1705426351003028], [-0.43798807560230063, -0.20689292471224427, -0.33372600380413486], [0.05668995489379499, -0.2931022193567212, 0.7532114084393808], [1.4388735938426587, 0.00020164295512421895, 0.3239119776314283]], [[-0.526514840115237, -1.2644927908197123, 0.5187849208779641], [-1.142518017652492, -0.7458563981883249, 0.35924465201120453], [0.40257336594453236, -0.400114751005071, -2.0192658104888017], [0.42051328729557524, 0.2595634588920596, -1.4123812154717754], [0.9520218679998016, -0.30075585250454295, 1.4367365400889385]]]], "dtype": "float64"}, {"__nd__": [3, 5], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[0.6568464180302901, 0.557511373399325, -0.5956318243301262, -1.7905532669844222, 1.1244767578841879, -0.588318953093688, -0.4717450752396646, 2.845645074509526], [2.127972196949781, 0.9537367314931017, -0.4554210238658131, 0.649461273978431, -2.2814697772878496, 0.10653270157752616, -0.759818281782311, -1.2856051436646596], [-1.5828044642378765, 0.5251415978751188, 0.08930037331340701, 0.1597236640001516, 0.8423760552565885, -0.8460837686653659, -1.2355844544689116, -1.7763653849182433], [-0.3351965739266949, 0.7601562085524842, -0.1494235252760479, 0.6411281903517326, -1.1810537136048151, -0.4691213318717145, -0.7659366813813416, -1.2483778102356926], [1.859473537260907, 0.8521407722678009, -0.7618005522771404, -0.7660172081897683, -0.057196853805184154, -1.4420539027685755, 0.43582716588443593, -1.2942387751632807], [-0.3704329748709035, -1.0248226241279634, -0.47984828973483024, -1.3311571805649849, -1.572822027026339, -0.9661408777268591, -0.680009857670002, -0.29488942970262855]], [[0.06898363214085088, 1.5268961098463931, 0.8122518305619173, -0.25289920168642466, 0.7259183347369238, -0.6888972645408504, 1.4484913866169449, 0.5494769624409097], [-1.3558393346715487, 0.755385523473681, 0.816092352617782, -2.685595289652432, -0.7480949542511323, -0.4738225196184616, -1.928512251800227, -1.0671774923126214], [-0.9483581881928416, 0.5070958053657446, 0.30046317773051506, 2.6261738286185947, -0.3942594033841144, 1.1532309938411287, 0.0788464753805641, 0.2709257789707485], [-1.1596298172861805, -1.9108997779424512, -0.7671482258172059, 0.7621685717133088, 1.5828056120254663, 0.6039910508652822, 0.49828090984319984, -0.8387668539743595], [1.534293845315934, 1.3438063203968533, -0.5308324871691382, -0.04637066919387357, -0.9960519234647858, 1.4672068853838773, 0.40635416902981664, -0.09506466916037241], [0.406900000422897, 0.440622290368524, -0.4114967034028557, -1.2415651857429464, -1.2837453687148983, -1.7383665131115047, 0.4612530204968294, 2.2185418386340072]]], "dtype": "float64"}, {"__nd__": [[[-1.6990692976365478, 0.1382874588248894, 0.41098394260608617, -0.9846256772614974, 0.20442580206739108, -1.0969368989183785, 1.2004021189881253, -0.5942490563115667]], [[0.10056038349126575, 0.6842415849945397, -0.4298193691923153, -1.7912308228410456, -0.531255658935443, 0.7629952676464102, -0.08453265729342355, 0.2297891839639593]]], "dtype": "float64"}, {"__nd__": [[[-1.0804018069049535, 0.14322291081554303, 0.7281669899188168, 0.03644802049061874, 1.9080145903201173, -0.2077935782605395, -1.0413502363373355, -1.6176592423461078]], [[1.1520058911926843, -0.0590182437964927, 1.0116217798624403, 0.45766891877208543, 1.6250842905757967, 1.2661073627611317, -0.3775545621683309, 0.2738849009141932]]], "dtype": "float64"}, {"__nd__": [[[[0.18905338179353307, -0.5227484414807474, -0.41306354339189344, -2.4414673826398556, 1.799707382720902, 1.1441658720372287, -0.32542283686782436, 0.7738065867276614], [0.28121066979764925, -0.5538228364240524, 0.9775674511260357, -0.31055654665915255, -0.3288239040579627, -0.7921467553588982, 0.45495807124085547, -0.09919805171738795], [0.5452887139646817, -0.6071856998706371, 0.12682784711186987, -0.8922740434297903, 0.8414649723701431, 0.18803508698068597, 0.33057100813532614, 0.41050391297026284], [-1.0107575001533344, 0.7831809961440773, 2.0567028183423686, -1.6384425032355252, -1.7294114671544816, -1.50483141386432, 0.8414588934539998, 0.12871565747406846], [1.078342440739298, 0.722430872307499, 0.21057181237528058, 0.28403814525037085, -0.16976049772313542, 0.8684602112115102, -1.1297159617807548, -0.4218588261783162], [0.2429388530987352, 1.8014208584493328, -0.7644641157203993, -1.0790604591369424, -0.5632871985040379, 0.9692722011381628, -0.23500550689182578, 1.324347019236957], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]], [[[2.3748869133171437, 0.2739322545923299, -0.2803823139429379, -0.7710521598195307, 0.6480646015444852, -0.19673006635772372, -0.17874637079406783, -0.1052597784150319], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.35411165760150265, -1.9683967522378658, 0.8992737076506074, -0.15824761527156533, -0.967681471724209, 1.6784190387621596, 0.7653546837661211, 0.04580783577937577], [-0.7454433887901278, -0.04336259016175258, -0.16392876295519593, 0.7247764935410931, 0.7980751811597928, -0.6677563797627329, -0.5493705607570516, -0.5320653277786109], [-1.3493172462579026, -0.5915748828912161, -0.09240806520639366, 0.6909519446056412, 1.3203794893667786, -0.8083458775593314, 0.5509765318813011, -0.44445682570554373], [2.0751877763153903, -0.048723544625703656, 0.5021269704086143, -0.9361621531378327, -0.8106894642994449, 0.20169530087063778, -0.3840544504961268, 0.3532885925705412], [-1.5618241192857418, 0.662004970148938, -0.8994803577797336, 1.7224233963662954, -0.28846724206797386, 1.0972650368859622, -1.394983322863019, 0.4289462445148908], [-0.8880296507945876, -0.4868890686859325, 0.024596382191250814, -0.3170749669346225, 0.26425562514648243, 0.7476554288062055, 0.6527821390898572, -0.08748148289345688], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]], [[[0.6679105062494222, 0.22100096632251784, -1.001333587233463, 0.3104215762864059, 0.3461775385567262, -0.8847733337672445, 1.0433118451954217, 0.5078871822065824], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [6, 1], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[-0.18783971905433797, -1.21984396644632, 0.4845266531778433, -0.07727260765760453, -0.2862305050128264, 1.0225523301343449, 0.5568437888324879, -0.06950984340533913], [-1.0134127141267366, 0.23870710497491457, -0.46317430716013713, 0.9387651486382901, -0.038759521046992165, 0.7023086671719226, -0.7494237395856685, 0.1653569365407862], [-0.5165184083422516, -0.0815638563896138, 0.31625516182538904, 0.18935061095383415, 0.7463907970304703, 0.1308079533896018, -0.6109762284194841, -0.6556396164223041], [-0.305146397195602, 0.03945601821866138, 0.0068202418275542095, 0.297637215016047, 0.1962338787333352, 0.30261153196359514, -0.5665262511661558, -0.15204702627684133], [-0.46179985546373553, -0.20268858696548311, 0.0665175003458605, 0.31159852056081877, 0.3298583898163181, 0.2189243884728971, -0.30932095663136333, -0.23838521249143696], [0.3293251420079213, -0.21938290724825318, 0.3170657824298309, -0.1558767727133298, 0.1149851281241165, 0.15063210100628718, -0.31748271206043766, -0.20483789800117697]], [[0.8847854555446513, 0.09555223476031102, -0.09952868111222518, 0.3763884516764322, 0.9191283363214476, 0.07882218172057752, 0.4067630582483526, 0.40305405537104605], [1.0985227491851388, -0.02808155725610976, 0.7892292904858706, 0.441400946026759, 1.483789922647333, 1.0284768021078878, -0.22057640827063785, 0.299737610172963], [0.9650633124709576, 0.04911646448502549, 0.23428103487526566, 0.4008065755844501, 1.1312102863956197, 0.4355041460439856, 0.17113984491176826, 0.3642493027493672], [0.8958265985983791, 0.08916561705463248, -0.05361762957125619, 0.3797468372823106, 0.9482973654851445, 0.12787900440574973, 0.37435624390580663, 0.39771698184585474], [0.9004527764291209, 0.08648965958359345, -0.03438115307680761, 0.38115398226371283, 0.96051902726461, 0.14843354159890354, 0.3607779680563836, 0.3954807775775889], [0.961195806763029, 0.05135357710197757, 0.21819925303187784, 0.3996301959798244, 1.1209929215597172, 0.41832045946149243, 0.18249134528049607, 0.3661187796886538]]], "dtype": "float64"}, {"__nd__": [[[[0.18905338179353307, -0.5227484414807474, -0.41306354339189344, -2.4414673826398556, 1.799707382720902, 1.1441658720372287, -0.32542283686782436, 0.7738065867276614], [0.28121066979764925, -0.5538228364240524, 0.9775674511260357, -0.31055654665915255, -0.3288239040579627, -0.7921467553588982, 0.45495807124085547, -0.09919805171738795], [0.5452887139646817, -0.6071856998706371, 0.12682784711186987, -0.8922740434297903, 0.8414649723701431, 0.18803508698068597, 0.33057100813532614, 0.41050391297026284], [-1.0107575001533344, 0.7831809961440773, 2.0567028183423686, -1.6384425032355252, -1.7294114671544816, -1.50483141386432, 0.8414588934539998, 0.12871565747406846], [1.078342440739298, 0.722430872307499, 0.21057181237528058, 0.28403814525037085, -0.16976049772313542, 0.8684602112115102, -1.1297159617807548, -0.4218588261783162], [0.2429388530987352, 1.8014208584493328, -0.7644641157203993, -1.0790604591369424, -0.5632871985040379, 0.9692722011381628, -0.23500550689182578, 1.324347019236957], [-1.6990692976365478, 0.1382874588248894, 0.41098394260608617, -0.9846256772614974, 0.20442580206739108, -1.0969368989183785, 1.2004021189881253, -0.5942490563115667]]], [[[2.3748869133171437, 0.2739322545923299, -0.2803823139429379, -0.7710521598195307, 0.6480646015444852, -0.19673006635772372, -0.17874637079406783, -0.1052597784150319], [0.10056038349126575, 0.6842415849945397, -0.4298193691923153, -1.7912308228410456, -0.531255658935443, 0.7629952676464102, -0.08453265729342355, 0.2297891839639593], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.35411165760150265, -1.9683967522378658, 0.8992737076506074, -0.15824761527156533, -0.967681471724209, 1.6784190387621596, 0.7653546837661211, 0.04580783577937577], [-0.7454433887901278, -0.04336259016175258, -0.16392876295519593, 0.7247764935410931, 0.7980751811597928, -0.6677563797627329, -0.5493705607570516, -0.5320653277786109], [-1.3493172462579026, -0.5915748828912161, -0.09240806520639366, 0.6909519446056412, 1.3203794893667786, -0.8083458775593314, 0.5509765318813011, -0.44445682570554373], [2.0751877763153903, -0.048723544625703656, 0.5021269704086143, -0.9361621531378327, -0.8106894642994449, 0.20169530087063778, -0.3840544504961268, 0.3532885925705412], [-1.5618241192857418, 0.662004970148938, -0.8994803577797336, 1.7224233963662954, -0.28846724206797386, 1.0972650368859622, -1.394983322863019, 0.4289462445148908], [-0.8880296507945876, -0.4868890686859325, 0.024596382191250814, -0.3170749669346225, 0.26425562514648243, 0.7476554288062055, 0.6527821390898572, -0.08748148289345688], [-1.0804018069049535, 0.14322291081554303, 0.7281669899188168, 0.03644802049061874, 1.9080145903201173, -0.2077935782605395, -1.0413502363373355, -1.6176592423461078]]], [[[0.6679105062494222, 0.22100096632251784, -1.001333587233463, 0.3104215762864059, 0.3461775385567262, -0.8847733337672445, 1.0433118451954217, 0.5078871822065824], [1.1520058911926843, -0.0590182437964927, 1.0116217798624403, 0.45766891877208543, 1.6250842905757967, 1.2661073627611317, -0.3775545621683309, 0.2738849009141932], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [7, 2], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[-1.1224991591932176, 0.9942165233017588, -0.6824805335261627, -2.587727904880445, 0.6099529550078873, -0.33607130533562796, -0.9213085668596558, 2.467170317451528, -0.09132832561026634, -1.3786239864782122, -0.6557888616330448, 0.33861865414829295, -0.26802132986016797, 1.1991018205283224, 1.0216524991877391, -0.8339216561974416], [-0.004582903023004578, 0.5990665963218542, -0.4553071190081083, -0.5063516509390211, 0.8370848619059076, 0.40846101505466353, -1.4448473649174036, -0.5911694464587628, -0.4556517912040103, -0.16039869132129655, -0.7598997316411998, 1.3000696618780292, -1.011799182073788, -1.1277618489488215, 0.16890502549989242, -2.295096218695581], [-1.072599754537817, 0.5848870562743542, -1.3659866532304288, -0.8850347184102482, 1.046369703549382, 0.4177032542376978, 0.018991127811624146, 0.2067489644779123, -0.7132614288572124, 0.2741943295287035, -0.23711467956631294, 2.466692737369118, 0.5068399491357682, -0.36892152256316235, -0.08286812839582022, 0.024504329152643865], [-1.4086029892855625, -0.8518156300754767, 1.4998194573598636, -1.43077829136212, -0.6069022987160904, 0.9984365827670537, 1.4870042012441271, -1.0472523972583951, 1.4401200916823877, -0.3845286445542359, 1.635937999528095, 0.48146398467144486, -0.30143934942313244, 0.7112938028139575, 0.5435286990993894, -0.7892229834355148], [2.1329989104025224, -0.35936242221827525, 1.1382944640819974, -0.4997788424323251, -1.8414428960639448, -1.1365460392416575, -0.11434961938701485, 0.6690422674050722, 1.7481711797623192, 0.06817593148376856, -0.29568118666229126, -1.2435652664649757, -0.05959330358389007, 0.28674176536500046, -2.227774261869857, 1.2913389834984377], [1.1238358178027428, 2.2635679776427633, -0.9252534535464243, 0.02425906603239125, 1.7177672167754252e-05, -1.3433594765064567, -0.8674712523946265, -1.0028725091508557, 0.4974759785799347, 0.4137368879910423, 1.331407254118111, -0.4222154509477325, -1.418299150993715, 0.41974276268766164, 0.3502131533117031, 0.6464753849372445], [1.2269155748136789, -0.6066330141765484, 0.778380901376294, -1.026530644519832, 0.13558100923796967, 0.6694432256814856, -0.13722004657307926, 0.19633407151678825, -1.4165822416638727, -0.6728836092202523, 0.12781967753018134, -1.2268057046995473, -0.8775528288167957, 2.412447242548542, 0.23366863445414107, -0.8519305764640945], [-1.0626219979151215, 1.349920236356491, -2.842635722702362, -0.407717255087351, 0.5435798336080095, -0.6556543860165066, 0.42508905693995763, -1.15343526821305, -0.12242576612565091, -0.06943148718295748, -0.10122037346198212, 0.02924674415042875, -0.08028572579931516, 1.0012797905611415, -0.3853942606183518, -0.7129018736271894]], [[-0.40711836980127253, -0.43082722504761384, -1.6526170276535996, -2.041986433545578, -1.4839376307571877, 0.15324482430053812, -0.7994988993760159, -0.5444471911031873, 2.7376985431237713, 0.9931286214269378, -0.34309127712400656, -0.7650974809801031, 1.243813254768777, -0.9154293574555002, 1.1042663154267118, 0.45491776532911854], [1.252765927337919, -0.8592309186145494, 0.7707863049512885, -1.9481503640032916, -0.30850432679042317, 0.9452505093827136, 0.5608177416586746, -1.973645777695202, 0.5494218583183289, 0.3177606128779975, -0.20235425229274503, -0.8623090670442447, -0.4205347478717994, -0.7138211763271045, 0.8081549615886352, 0.3269564631002771], [0.9301929345388833, 0.5505679669966813, -1.4225020276599285, -0.003648102084038217, 2.0051001451105592, -1.2512077570376248, 0.14084791046927592, -0.9215608059520919, -1.0304752960600443, 0.07115548261858266, 0.7788941721039077, 0.6279189559241043, 0.32190530176318083, 0.8101217629425644, -1.771784895273859, 0.6765504710111022], [0.26267164971807505, -0.3258240042586636, -0.1528144653616176, -0.9964066946896972, 1.0230422283267584, -0.8969496711866208, -1.6199879787569698, -0.14953769763507402, -0.7231910433422903, -0.21331830388227668, 1.3711320995123473, -0.28041644234240426, -0.01840154940527485, 2.127685343194198, -1.5467555517496037, 0.22761539285093157], [1.0300360917397358, 1.3790245742629843, 2.1122492458120354, -0.7967726389955628, 0.4749739115706815, -0.23072933420675368, -0.5896416570631288, -0.8614416605478826, -1.600472157009654, 0.20201597501551757, 0.6228167213919259, -1.1983242953649467, 0.6320664939408687, -0.6868362541656808, -1.1491180842934814, -2.7654160197902855], [0.9779204556171887, 0.5900130410293513, -0.44342758934234394, -0.5181060641881448, -1.4363276567654457, 0.39181699732382275, -0.19110986260805823, 0.4171360205961122, 0.05362837639264533, -1.539796823809087, -1.6890698669024233, 0.2998860520849562, -1.821377493204597, 0.1572042966227879, 0.7969984837700558, -1.4054841569743521], [0.5694829284100298, -0.06115038988089331, 0.4366601039768501, -0.7351228367784193, -0.10857977740435112, 0.3405254255300457, 1.7564114701292186, 0.6585507878644343, 0.424056492220449, -1.3804391245295171, 0.26395432386215806, -0.7550180054732525, -0.13604215904887154, 0.20719158846184046, 0.9158821084206138, 0.5392610362430644], [0.13434144598772765, -0.11184676527876072, 1.3137245154052999, -0.9425021016246354, -0.009123697316540832, 1.338755310308567, -0.2414776309049401, 0.7907440624428624, 0.8529774749552903, 1.950031159400373, 1.0958369500323615, -1.7058370592703518, -0.04420303946135186, 0.24934324079027947, 0.7854777605275928, 0.34012297364097027]], [[1.8472790560549168, 1.054984669674182, -1.2480686294051366, -0.700604144415385, -2.146016366850243, 1.6737766240484186, -0.2036363198327749, -0.22646598624295106, -1.6842881116295754, -0.5421620210728261, 0.3248449319902771, -0.5238729378012491, -0.5194654956971493, -0.7537512087502728, 1.8672085535614877, -0.6957301294935373], [-1.6134731506998456, -0.6759339950063554, -0.51354197524934, -0.2606045945792195, -0.4185172944041889, -1.3075921239748125, -0.3830552438438988, 0.20704557738079865, -1.6059888174916972, 1.8144290854914367, 1.5321985750119627, -2.030519614692431, -0.5649584954644857, 0.8303572399257155, 1.1683423264654813, 1.6183461407748712], [-2.0661846714422185, -1.3190441251803353, -0.7134859492885108, 0.3667945330226859, -1.2806649763610893, 0.2523663868170364, 0.27537778525999035, 0.5494953836646462, -1.4874353097861168, 0.02401883720400528, -0.09884186188503467, -0.5925266255898535, 1.1594328211434304, -0.45735250411490924, 1.315632416509579, -0.7565046825944916], [1.584128562015884, 1.531626786578025, -0.9746176954636457, 0.20796228166789654, 0.18827287377077315, -1.1192101978762619, 0.6750581373284622, 0.48549656374887296, -1.0000772680216847, -0.011015205116916914, 0.7970510731520428, -0.2660528347661754, -0.827451794342119, 1.5677791404299568, -1.0920723071365355, 0.256062055828496], [1.0545179094195678, 0.7209298491207806, -1.9468703320403449, -0.4081268152247772, -0.2066330208017125, -0.2830014179704721, 0.5728076020832752, 0.4919133266481511, 0.08075903915343023, 0.2186379894362371, 0.9812970728920072, 1.223767005564508, -1.9649516465503092, 2.179419935131282, -0.5958083787043419, 0.7622644638316202], [-1.6825243984654168, -0.5499566738047589, -1.3022358227708337, 1.0799956046121186, 0.8791921566084736, -0.014258561106073918, -0.191621763653623, 0.3743670435807552, -0.9717922399428903, -0.6341529605734781, -0.7940674495375496, 0.8150408754310717, 0.009890737703085203, -1.3101971471806988, 0.33402886347538685, 0.5182516284039733], [1.3930190463488872, -0.5800811175743941, 0.39561386467358356, 2.098142607409753, -0.34563837749835363, -0.3843082907125673, 0.5018574376979424, -0.5246061354052393, -0.014510304698945535, -0.528049017216281, -0.8127743293684287, 0.8738355654988246, -0.3993484373183031, -1.0392724913428106, -0.3114653362673359, -0.16472668350862385], [0.2824281991980708, 0.9269111835727383, 1.7735436622238923, -1.2672785037066503, -0.2756551445332441, -1.8735724798986586, 0.1939528994633171, 0.4982829807792664, -0.6966447831421315, -0.9639397566443456, -1.045215516890933, -1.5603387529823856, 0.5948299097184745, 1.8879981977842317, -1.203126161998486, 1.6032475190162354]], [[-0.16928573215030954, -1.4659799150409802, 0.5120036607899497, 0.7078411390828067, 0.3362567988666745, -0.32899067058102455, 1.6249445261970499, -0.35567022570187523, 0.5742907869152626, 0.5873076172497034, -0.97993348196342, -1.9494042510054572, -1.8300852234137897, -0.48046889699998796, 0.28185455869336173, -1.8850449501581044], [-0.8135587239031288, 1.4702819391858084, -1.9059646624644635, 0.38231245645325557, 0.26897591074410615, 0.7102467223354261, -1.5820321345172357, 0.2505845965415184, -1.5030516417780693, 1.2591683071733701, -0.9711245328684528, 0.28988387093838164, -0.9073076701077541, 0.04323318559981589, -1.6539282150672092, 0.18957813306831867], [0.5127954361185663, -0.11008884168514857, 1.5392089194402008, 0.2748983144803266, 0.6973913260098954, 0.41405187233478874, 1.8035929032779516, -0.7651493395661518, -0.37227710607023035, -0.3194012040231828, 0.5307744972087725, 0.17046454783398943, 0.7538608773850645, 0.20826906065756592, -0.6144510127512338, -0.15290267840628477], [0.35579816691879107, -2.408973854465761, 1.3274888155525175, -0.6586859435780759, -0.4417264005057006, 0.18642425775105692, -0.3307605012087098, -1.5513845269333426, -0.30464950212418734, 1.3409161006224404, 0.9983112816762165, 1.1928827859743476, -0.6232307780311632, -0.279404692242727, 1.0149625308723331, 1.457461637327464], [-0.019024247211218915, -1.0349906277294474, -2.014125406838725, 1.4032734050501081, -1.5934399251309603, 0.40073558069138, 0.8737286994852778, 3.2936566878551727, 0.16579816385404184, -0.142744069224455, 0.44770113054762395, 0.5021828155072129, 1.3561495193874011, 0.6894013102685443, -0.7972708166114919, -2.1752103369166043], [0.29329759991058185, -0.7041776777464693, 1.439232297891831, -1.356228377961351, 0.20056940727506306, -0.7563377363990464, -0.9060184084737523, 1.7109003176319157, 0.17168456302644972, -0.18815480502836673, 0.7606294890873249, -0.8097084889696944, -0.3796728278404442, -0.6578619425206957, 2.042267417198696, 0.33993504187245044], [0.6481192559454834, 1.150268842591634, -0.19239802254704683, -0.16039804110040548, -1.6907525969549917, 0.4931538377964738, 0.2656588879647659, -0.42733569671207866, 1.241978905204404, -1.8489666404387919, 0.4308423357882474, -1.4276734431639235, -1.1860284296971704, -0.22169237096467545, 1.3743988780261713, -1.2347181810180048], [-1.038430824755351, -1.2581256528562743, 2.079166372762694, 0.5366870861917691, 0.4153254631293544, 2.1341355970917864, 0.2540054344722237, -2.1830615926901467, -0.5280474011759992, -1.2447540402470687, 2.095244749291541, -1.472980509874603, 0.31001811340254026, -0.5396205708959053, -0.46294301093723766, 0.016050163909523987]]], "dtype": "float64"}, {"__nd__": [[[-0.14544487154245192, -0.7164522571081597, -1.28919322719092, -0.6830203324000345, -1.0910181622054855, -0.9495899613521656, 0.114886001535766, -0.16705388559636833, 0.4878608415208204, -0.25653410585141756, 1.9879044407503788, -0.7698174393859692, -1.0494574911793721, 1.0539101395445558, -0.7445954455878293, 0.11776109258766718], [1.0053015668910137, -0.1142963269697127, -0.38148103401332834, 0.9936009708862976, 0.33223679954700147, -0.5661107688153971, -1.2590703332499047, -0.4079930775147451, -1.7380189293092558, -0.9224034990272185, -1.1545697150335459, 0.28946781577853076, 0.7758850590296091, -0.6944417419171223, -0.47600169088170596, -1.163570066502072]], [[-1.4558716071986488, -0.4699931875923203, 0.564651281355481, 0.46586731042272617, 0.5901614337728481, -0.3059991891543134, -2.8606225989825966, 0.941873547171145, 1.3517080423461578, 0.9267686892688831, 0.9187685992060648, 1.1214500855429523, 2.769683287961346, 0.3209113093482113, 1.6048923457166533, 0.8318099269703514], [0.6970982940252463, -0.14309223040428232, -0.3627842298347233, -1.1739196317090077, 0.604586444674966, -0.32904182427704876, -1.2511243572359672, -0.7932021320713456, -1.237353184050747, 1.657955572756555, -0.2588428600301334, 0.10751022536384264, -1.2631742271058926, -1.077707360000256, 1.0853383975503472, -0.2482160614062073]], [[0.2267708786188296, 1.903719540005617, 2.1684157123550465, 0.7132322831037413, 0.504002187870548, 0.24362088066698953, -0.2981776943387448, 2.1019305233624266, -1.2637991429270883, -0.09444450704511713, -1.9996431856170793, -1.3159613447991658, 0.4179761229367063, 0.4737956461830622, 0.34948936620293497, -0.33738650439041584], [-0.7354851140141753, -0.5605398965196526, -0.39853710833038963, 1.1365535742141335, 1.5150308036745619, 0.45937210573959025, -0.5600021510349511, -0.973885944987591, 0.3160481273078601, -0.23779689710483434, -1.8960106100940826, 1.424341360634964, -0.008583035425387036, -0.3314610784330754, -0.9046298820827241, -0.9086011229733484]], [[0.9341038038726168, -0.9809394388777001, 0.09944872702975784, 0.6594175075155629, 1.916430660845645, 0.6280729155292678, -0.035543979190243234, -0.4572617685607399, 0.889372065743123, 1.0439059502274242, -0.44265502454534617, 0.6105468435269886, -0.14369760838662038, -1.686521242221457, -1.046413692527123, 0.8411470639005992], [1.4986090579948839, 0.7053757558894472, -0.21011368424475652, -0.7444566477548312, 0.5577568332992092, 0.34395172820801173, -1.681218971972077, 0.6770060021095777, -0.7349183462739384, 0.698344279432918, 0.558303354154615, 0.7546725929699328, 1.1140404809054816, 0.7328285620708094, -1.0901414224087804, -1.0578753954286788]]], "dtype": "float64"}, {"__nd__": [[[1.0810468379366278, 0.49670085743436004, -1.6323219323519624, -2.084963671876908, 0.9300367521567386, 2.3133719216161226, 0.8620755189271605, -0.24423687921083367, 0.28050634678012704, -2.828253250623994, -1.0346619100677745, 1.0564668867073979, 0.898156332357861, -2.207805125910652, 1.232281008101371, 0.5591688855045999], [0.9769022626482755, -0.20694513814567128, 0.007602143792902422, -1.116946285492455, 0.33671129595538396, 0.9348941188848842, -0.9742362659018642, 0.3841855024216996, -0.5492529925296089, 1.0872433194559101, 1.0224074110353782, -2.4502804893865626, 0.4741224606665683, 1.0586367880855143, -0.578971825502438, -0.22136518586369025]], [[-0.15436727838762757, -1.187741910139558, -0.11791444842706794, 1.6200151078549272, 0.4992446552165796, 1.586752076795199, 1.035065737434756, 1.2518367668529706, 0.13091240011652536, 0.8099439569844279, 2.1002217943852965, 1.534742697196371, 0.34011868915030824, -0.7217455935489808, -0.9549883153627023, 0.2537528581669864], [0.46012695162139583, -1.9045817342802513, 0.23222421906122503, 0.36513287412683576, 0.318409458566481, -0.5668580127420708, 2.398263544594866, 0.9107897718504723, -1.4228340147390692, -0.7405895903642278, 0.8339025111981764, 0.5329341183962628, 0.13188270642753452, -0.03434879399637327, -0.22626784101888814, -0.47137749319831695]], [[-1.072822420682492, 0.5879554915597687, -2.898614617024146, -0.774015279292408, 1.2674952300868871, -0.8363261869699357, -1.2632614362953578, 1.9392464493678783, 1.0507760296067028, -0.18549136253838838, -0.6394923529564971, -0.27503010905511827, -0.39684249843042757, -0.2885423471166199, -0.804998969017241, 0.5216492324305687], [-0.6191295508246943, -1.9232642651625773, -1.3860513922345854, -0.04642735433419728, -1.1060513779491727, -2.6173164194237817, 1.5012597319609273, 0.4586703067977707, -0.5503271136299331, -0.6016890085218324, 0.46896230646529885, 1.3587476828431804, -0.7107702239060719, 0.3593028784122634, -0.7150284735470539, 0.2684755280813338]], [[-1.2490682262713597, -1.9662644726986416, 0.017614937041686207, -0.1395605118820667, 0.6751620988202943, -0.7658265377654894, 1.3478297548091558, 0.024913568219149716, 1.524070517801729, -0.42700390998073806, 1.4880909584078001, -0.14079386165808747, 0.0726676372661932, 0.9252985108569339, -1.9058651257910082, -1.1777333935736278], [1.826289676185761, -1.634458097876963, -0.5896575671954235, 0.4900469922667428, 1.4412212133691968, -0.5967857202585577, 0.06297177296515619, -1.241494570113807, -0.46837959642252563, 0.9917968560582078, 0.26963185086493774, -0.10051772812703341, 0.3737700871414614, 0.07718806994273542, 0.11133663404623863, -0.42843394393473433]]], "dtype": "float64"}, {"__nd__": [[[[40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]], [[[40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]], [[[-0.4276300729183073, 1.3049469505616562, 2.570092738692354, -0.7399058822942611, -2.174502072773742, -0.4254398963296527, 0.3768451073305704, 1.0501313269503614, 1.0790155135012853, -0.7839513663149491, -0.7932353410041735, 1.2673009187104822, -1.0923877024313569, 1.0902313512146344, -0.2268879272034576, 0.7664035884711488], [0.7061647137637838, 0.7766524810610778, 0.8204266971936063, -0.9893909801613543, 0.29752110861128966, 0.013982411315735277, 0.30840802864763195, 1.2332483942406183, -0.2711082411279299, -0.7119704226209977, -0.728115476185683, -0.31081837778290006, 0.7526105507610801, -0.8148156257479607, 1.1592578951012953, 0.29045146553250784], [-0.22611092064531074, -1.3316509777862344, 0.24493318448510143, 0.6914693220924303, -1.7657657867263103, 1.1919595712856506, 1.647747014893282, 0.338502296909925, 0.4917808983658258, 2.4136949965880823, 1.2318906798511962, -0.8977571989333307, -0.01664518614742305, -0.15693106761722703, -0.15087679609514876, -2.3642678177611556], [0.7570048405627458, -0.6367361170630729, 0.16365146447771292, -0.13613934374199768, 0.8710953770180234, -0.29580869027763773, -0.7954280607901872, 0.5324224800592481, 1.4048856848402156, -0.3198106833867411, 1.3699480278262324, 0.2621831500067624, -0.5327294852810801, 1.5251578003581547, -0.818626335553522, 1.0140481991147188], [-0.4981859667883459, 0.4502186456421907, -1.234148305277474, -0.8375926212541703, 1.0650523006996653, 0.18811685191459512, 0.03829301646633128, -0.43980824203442026, 0.446480673727825, -1.3972692775978013, -1.1199352443263735, -0.157790365329733, -0.8342810311785638, -0.7005937683055597, -0.07150325804844848, -0.13752133194645022], [-0.222506700637484, 1.6652756190688167, 1.0110758002746831, 1.2297345185288913, -0.028162001365168308, -0.5046842878231101, -1.4538991774513217, -0.34813740474730365, -2.085955516806786, -0.5837413170812261, 0.764238935433696, 1.621530244280291, 0.9656069166841026, -0.8616004422818208, 0.49261852280236235, -0.7891640250574642], [0.8229737900880982, -0.11136118318513945, 0.8582001144570633, 2.572895572233041, 0.9225243033323701, 2.3324499453787046, 0.177442312981226, -0.32971772068132815, 1.3337043892100016, 1.724356555267187, -0.5550145945569381, 0.07663196559606328, -1.4345295287904392, -0.20004587761769155, 0.8883322285022602, -0.08638483740324072], [-0.569220157581707, -1.407239902207787, 0.36825449991781684, 0.797540663665427, 0.5965006827757677, 0.25531569894951217, -1.2740653493386656, 1.7618086407811855, -0.05653636459299152, -0.14479645001033087, 0.7329285500357282, -2.163186235204895, -0.0644672373325154, -0.13678960297985446, 0.3722831319448136, 1.3471598477745907], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[0.23207021517525864, -1.1791464394934508, 1.3586400261294171, -1.2926229107419693, 0.25441917324395535, 0.13490564635114768, -1.0507245554685565, 0.8869794016080271, -1.415958011229726, -0.683484532933555, -0.028702202916173543, -0.41249769672963527, 0.46780667461364833, 0.9240548410021371, -0.29377376425342144, -0.5414659678720907], [-0.5169782292637785, -0.3655939450055468, 0.8096488946941695, -0.2175850065483592, 0.40763953366453465, -0.09115277603330242, -0.844465175412494, 0.8637197398257774, -1.0380412580682665, 0.006197796707115902, 1.4870238550758998, -1.0931070420832962, -0.05518852081126953, -1.5847810984165838, -0.5094162712223194, -1.4945996585123718], [-0.3615871788340143, -0.5382974610574749, 0.13197161380603412, 1.5843009073214365, 0.28298451661725926, 0.6581744527499446, -0.05070193949372962, -0.24452361546620502, -0.8339818243786973, -1.800856820932183, 0.06862036839919183, 1.6291235191475462, 0.4762998718707188, -0.5339889586362624, 1.6494061436322494, -0.4326235019616808], [-0.6188397120633242, -1.212694351698304, -0.962050935083037, -0.2463628695717934, 1.999020667820133, -0.20617621326392488, -1.0315163014000943, -0.605418629431135, -0.2228617607558514, -1.7117519241888182, 0.12662843610718544, 0.7006684855746799, 0.3208122410050434, 0.6007326507656642, -1.2895470431032947, -1.6985919026466894], [1.2855745952249331, 1.0943313362419658, -1.5402644573983837, 0.20638165811738485, 0.025527869974148333, 0.5448862967059741, 0.20858621210957912, -0.5021319045882552, 0.19525663814289484, 0.15704688166820915, -0.0486276310087608, -1.8524186305219243, -2.995201022440529, -0.7901887934234801, -1.6347715081124043, 0.25256204274779664], [0.7160967175077158, 0.2631154237292073, -0.3695278958798835, 0.07048084459204089, 1.167341045550715, -0.7288080336039454, 0.9222947397293393, 0.7076163309240973, 0.2616842740988124, 1.4519301576671035, -0.7178185832067028, 0.9425060224272724, -0.9289515855680833, -0.7670315765631621, -1.059711085029156, 0.44583159760427166], [-0.9397600370828224, 0.05215989851629757, -1.136711911629779, 1.066257585784627, 1.8361735112481172, 0.22607664232280375, 1.6433654157233566, 0.5606636371484345, 2.169309982212192, 0.09971426544292412, -1.5704236041028392, 1.49058403082891, -0.12636714234675797, -1.0358450164480129, -0.6562658594674428, -1.6263317217708266], [-0.16770180360718803, -1.0712637551480848, 1.484180556178205, 0.6644919985471294, -1.6604060196052313, -1.5093723248953304, -0.5669757804700066, -1.9250410205953503, -0.10980580857164887, -0.04344814976476555, -0.7728063339237182, -0.5464943802106043, 0.498852844163285, 1.4216846227430395, 1.4956732179240155, 0.03430883063776426], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]], [[[1.4038291578515893, 0.5568982661709865, -1.3240953569990666, -2.201557117006762, 0.09988180022201963, 1.1820845467832928, 0.17374005455439948, 0.8820660345952854, -0.4846734103640801, -1.3639205623940929, 0.7485937326351536, 0.09894227464054638, 0.6836829270923552, -1.3815659418907666, 0.1815534413430963, -0.6255009899606692], [0.8065888431513611, -0.5806873242375763, -0.6073949019765076, -0.5857203770133427, 1.263717038467993, 0.028864195779211843, -0.7599971570164401, -1.0225841755527632, -1.0441224416609518, 1.4626275482563855, -1.814774513497246, -0.8503575320650164, 0.6378903416209587, 1.1784957119103587, -0.14827740739445017, 0.7855160537736622], [0.7596172911688887, 0.9861870768741756, 0.04641503077493213, 1.6549681110403118, 0.2161253725997985, 1.5482000856724596, -1.082788417359302, 0.96061305531659, -1.2350369202277605, -1.4755833724170708, 0.5254990763360979, -0.7997444816067071, 0.47699373504770903, -1.0392571047529968, 1.6403754882158161, 0.04541300736884309], [-1.0942737499358477, 0.21565910210810485, 0.9240799367594777, -0.7268860672016146, 0.5617496770275349, 2.093347711888077, 0.9154857742071912, 0.039247960186757934, -0.5195256010390428, 0.6291380372130817, -1.4644056944900845, -2.986478022301571, -0.613397013531575, -1.0284662209338726, 1.799839710587298, 0.19588654490386795], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[-0.0909370888182374, -0.06102960599713808, -1.0791675863890429, -0.23909435282450797, 0.28724736695517095, 0.5837607504782176, 1.295664503057182, 0.057048197180256645, -0.4357512542885165, 0.427944847082082, -0.6054584854017475, -1.2284018657621247, 1.1658174133660713, -0.35966252750539546, 0.14188858466737753, 0.6701044983880885], [-1.087502107535094, 0.6610705932032451, 0.47000781019778404, -0.0786951965212257, 0.413828570903945, -0.7375753310221359, -2.186627825908461, -0.26442112461867023, -0.8926158968589242, 0.8870077749338153, 0.282840295701735, -0.511611700526905, 0.0655950955625179, 1.3710853715756686, 1.1530663941817965, 0.386902226933139], [0.4485263830803767, -0.2604080417729926, -0.11286324321516372, 0.6066693824226794, -0.10218464932996439, 0.31613473140693815, -1.098923752533973, -1.652323167168886, -1.6826912560069485, -0.42126579609714815, 1.3442906422568701, 1.1995958076434512, -0.9814296623325831, -1.5408069593316496, 0.8088977952524207, 0.47898577886724764], [-1.3532677622094842, 1.0735628380399151, 0.19024643307954248, 1.3394518668961404, 0.3579726340873901, 0.9059391704119965, 0.3368444827337588, -0.9376466937313434, -0.2178904256607671, -0.6630188479906014, 0.8452523143858375, -0.9817645515074065, -1.51943281101853, 0.6968105828302908, 1.1609312676187333, 1.4599091358239493], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]], [[[-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]], [[[0.45348924855482103, 0.3490698320618196, -0.14843380445321855, 1.241103706461563, 0.02043960560289747, -1.177624501666097, 0.2978175835368062, 0.12318903722818446, -0.040502829002204206, 0.23232640681619574, 2.3495531721956855, 0.23351901997783128, 0.457794753077021, -0.24645480856286245, 1.3103865519597715, -0.05656058864668221], [-0.8021083355278161, -1.2226428849539965, 0.5270918831485563, 0.7442358784331412, -1.588551827774419, -0.7237141069722046, 0.6413868499110363, 1.0271160617787196, -1.3212260599450445, -0.4087118552942355, 0.831768869291634, -0.5348942725265395, -0.4517353828944098, -0.5417974689822959, -1.3082161382774413, 0.04540970767256627], [0.3029250715030712, -1.4035863277668814, 0.09848556594080901, 1.1060581160781187, 0.20567148048059108, 0.9453840155840322, -0.5513148318244018, 1.3672061393525128, -0.7881364875171292, 1.8833833034706247, -0.4511039082529584, -0.7207785056850035, -0.8949810818089227, -0.41504493138051307, -0.4955772466526089, -0.6917725638406813], [-0.33178500902044955, 0.6620391064287441, 0.8797226429138996, -0.3400291332547872, 1.2321465640163338, -1.000674009730229, -1.0312794134437753, -1.1338053955310952, 0.5986278853586913, -1.5538412220643407, -1.399834668218403, 1.2750935383100084, 1.7088872697564919, 1.3391127644647582, -1.4963747550383706, 0.0065392554304298115], [0.826159701872232, 0.09072638217942289, -1.6242270588344543, -0.3487217781640566, -1.7051393371808987, -1.2394411568718704, 2.1451520357429232, 0.6719362710029061, -1.4750693303909352, -1.5086182003094526, 0.4481253905037407, -0.6179530470677405, 0.5686865679295522, -0.6563408232622712, -0.20918554944812545, 0.22913804836699356], [-1.0425843086187383, -1.930194037646372, -1.204502626035393, -0.7088424652129783, -0.21507838266500906, -0.0706887691553342, -0.07921269392390515, 0.5019532202178966, -0.9050357596317037, 1.382136313697382, 1.988347489154716, 0.3520614266417755, 0.1739412726734303, 1.2332775946474066, 0.6337528708044734, 0.17293903103560176], [-0.18279434902449157, -0.7540787671055834, -1.1843869781421177, 0.6499515036791333, 1.17223424297214, -1.0725984514356413, -0.08787476349616176, 0.2021257572271039, -0.44147812189575236, -0.7774019533764328, -0.40872577854715136, -1.986573793588022, -0.07726774500169377, -0.3106481701334202, 2.1002647416741365, 0.9197710972652077], [-0.36233625364362343, -0.02498371325575958, 1.6989366631605227, 1.588795013006287, 0.16741353315593088, -0.1457453698422363, -1.46231693200154, -0.34448553044702235, -0.18632164207772312, -0.7531993282122036, 1.7125871847761867, -1.8930187151158349, -0.15312954802751536, 0.3895240579324037, 0.419251660827471, 1.6691079751612958], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[-0.43407506437830745, 1.252468159533808, 0.2040317137638252, -0.7745705329496348, -0.797057565340283, -1.195201052013818, 1.1866185535233382, -2.040504380600246, 0.25173079304344975, -1.4622981361005167, -1.0827615837006996, -1.512913408289115, 0.9379880811278674, -0.10783298845229917, 2.7162272789621746, -1.064976433074287], [-1.1233488174763162, 0.5610081682206371, 0.5322913211026215, -0.312752579841828, -2.052834922739572, 1.4116051772766451, 0.5343497679978567, 0.36350691367384036, -0.6297993974836996, 0.603452291181055, -0.5992394254479873, 0.002782362028699419, -0.23031125033700295, 1.215344509742451, -0.5142291775100767, -0.39040138695912274], [1.0593440260021498, 0.17986184518242124, -0.06755071567350249, -0.2900110919043919, 0.9010508614757021, -0.7011904877438835, 0.4197467627090478, 1.9785337005133665, 0.34653185158580296, -0.8393389532624606, -1.2570782775538027, 0.0913928921761727, -1.2842551323548932, -1.6518002934310132, -0.8178593240212917, -1.1548838798059364], [0.8153216554309594, -1.5129322089603954, 1.5206858984910974, -0.8151857651342108, 0.054662978872448975, 0.6076080292332172, 0.15800535486330722, 0.6552466645712847, -1.054365703737103, -2.341079099388513, -0.270742250497455, -1.7528945955959814, -0.6525186158748978, 0.8720540969383103, -0.27137987374942657, -0.6681801860042257], [-1.205816597844541, -0.36867168433691105, -0.25081680293783243, -1.6610216668350122, 0.21573514740291905, -1.1555926876966116, 0.13841683858152587, -0.532583247535998, 0.9094871454462922, 0.775747641036947, -0.07191889499725981, -0.18943499544916995, 0.35863452374958005, 1.3057222449014558, 0.9909160243519445, 0.08859570606956685], [0.24926498110087889, 1.5156350108956544, -1.6955165561987913, 0.4848982329631039, -0.9567307938507938, 1.3039183586818974, -0.5436124326602008, -0.15811483591241962, -1.13066503564093, -0.024170144096250457, 0.7569795471893374, 1.6395380071343677, -1.2954630823226267, -0.4699377462582708, 3.8864552710410796, 0.9217358897855625], [-0.025641660071995545, 0.5070818598060605, -2.681901571333781, 1.60351971804367, 0.14751459445916568, -1.4456269739712746, 1.1621213844890788, -0.6370449948071236, -1.0342037330804474, 0.5195441639314857, 0.3987098615681088, -0.46217542782828724, -0.07581267564668939, -1.3466926889404711, -1.4483717961252065, 0.8991607563497701], [1.3172838676535006, 0.4312852345935342, 0.37363965609735533, -0.3960975965738327, -0.13019312759828652, -0.9373248956610112, 1.758270921248861, -0.5680427060002515, 0.41060240157624794, 0.00016674677896742597, -1.4002809519865258, 1.9391935035563979, -0.3758299307032596, -0.7687258613081643, 0.11466400875383675, -0.8982965864809159], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]], [[[-0.05049641095083523, 0.5057614826048096, -1.1841302440536672, -2.098577562735211, -0.00016556387578946797, -0.4367785013228669, 0.09812105876600097, 0.6559041690469416, -0.2033843250993628, 1.2112682825036059, -0.024050770185300285, 0.13227414545791594, 0.855094960218829, 0.6115973107692956, -0.06459700496052836, 0.15147794717976315], [-0.7325077075725053, -0.5885402327147731, 0.13132265828990042, 0.8462540265376409, -0.4059438232776296, 0.8735382871450506, 1.9794363955328722, 2.1877755810645407, 0.06357245403903584, 1.095914334286998, 0.652855865054215, 0.8306879240179876, 0.884154477405213, 0.570092253775361, 0.875439245116228, 0.24068192177543613], [-0.20984597803395438, 0.30575821533983477, 0.5470753539463475, 1.28047588518512, 0.4263070672726552, 0.7989929010721081, 1.4833457972471682, -0.8798562848592036, -0.20736754043443523, 0.5968579591827916, 0.08149553187709808, -0.9605385504148742, 1.0465215571750213, -0.015999235517005012, -0.9484673608574571, 1.9530872597100337], [-1.232303911515722, 0.03862752955704673, -0.6207221489901279, 0.16117169074175516, 0.6161871156697357, -0.7686342067399412, -0.05466770264738846, -0.08203095697633514, -0.9570086194403548, 1.216826196312125, 0.22016793539333407, -0.24215134445472328, -0.6210333963155036, -0.28904388032810574, 0.18836057398498784, -0.4175319404494895], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[0.8960789585932066, -0.40792916481990404, 0.9048473795644835, 0.5957071916576799, 1.4620180665716653, 0.06936802967455195, -0.14207718027474814, -0.4148282000124647, -0.2837981802135376, -0.6733856307956471, -0.24865063620055392, 0.5934916895880746, 0.5493718052575148, 1.4085243644565233, 1.1823715292118242, 0.7123072141644236], [-1.0688020852532705, 0.6528002570395969, 1.2463039776292244, -1.225101651231791, -1.0397617546110227, -0.8794495187402187, -0.9670960473664579, -0.21878888911238828, -0.8675790226551632, 0.45446221707678186, -1.1485436386065973, -0.06447743418482642, -0.4381052537646047, -0.909855889510612, 1.0947980204499768, 0.08675415760012148], [0.6002408553014308, -0.27574187646611276, 0.7611515544215757, 0.6721111673832539, -0.09537194545221389, -1.717910238177568, 0.8777901325375698, 0.7497206522219166, 0.23805901725218487, 0.2999132827833922, 0.18103458729481264, 0.2500183974042268, -0.9478056260663629, 0.001718988139928557, 0.37575238954691337, -0.5547694668570979], [-0.9129834258945715, 1.9173918762969722, 0.6651930159072552, -0.24874803952609156, -0.434376402431708, -2.5850457856618068, -1.3675827684394424, -0.19130937784917285, 1.3467440200733614, -1.9998259095826407, -0.6417015069465, 1.9730949242733973, 0.7285887918640518, 0.473569638666792, 1.2539915649475473, -0.06093714534384334], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [0, 0, 8, 4], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[1.0810468379366278, 0.49670085743436004, -1.6323219323519624, -2.084963671876908, 0.9300367521567386, 2.3133719216161226, 0.8620755189271605, -0.24423687921083367, 0.28050634678012704, -2.828253250623994, -1.0346619100677745, 1.0564668867073979, 0.898156332357861, -2.207805125910652, 1.232281008101371, 0.5591688855045999], [1.0810468379366278, 0.49670085743436004, -1.6323219323519624, -2.084963671876908, 0.9300367521567386, 2.3133719216161226, 0.8620755189271605, -0.24423687921083367, 0.28050634678012704, -2.828253250623994, -1.0346619100677745, 1.0564668867073979, 0.898156332357861, -2.207805125910652, 1.232281008101371, 0.5591688855045999], [1.0810468379366278, 0.49670085743436004, -1.6323219323519624, -2.084963671876908, 0.9300367521567386, 2.3133719216161226, 0.8620755189271605, -0.24423687921083367, 0.28050634678012704, -2.828253250623994, -1.0346619100677745, 1.0564668867073979, 0.898156332357861, -2.207805125910652, 1.232281008101371, 0.5591688855045999], [1.0810468379366278, 0.49670085743436004, -1.6323219323519624, -2.084963671876908, 0.9300367521567386, 2.3133719216161226, 0.8620755189271605, -0.24423687921083367, 0.28050634678012704, -2.828253250623994, -1.0346619100677745, 1.0564668867073979, 0.898156332357861, -2.207805125910652, 1.232281008101371, 0.5591688855045999], [0.9769022626482755, -0.20694513814567128, 0.007602143792902422, -1.116946285492455, 0.33671129595538396, 0.9348941188848842, -0.9742362659018642, 0.3841855024216996, -0.5492529925296089, 1.0872433194559101, 1.0224074110353782, -2.4502804893865626, 0.4741224606665683, 1.0586367880855143, -0.578971825502438, -0.22136518586369025], [0.9769022626482755, -0.20694513814567128, 0.007602143792902422, -1.116946285492455, 0.33671129595538396, 0.9348941188848842, -0.9742362659018642, 0.3841855024216996, -0.5492529925296089, 1.0872433194559101, 1.0224074110353782, -2.4502804893865626, 0.4741224606665683, 1.0586367880855143, -0.578971825502438, -0.22136518586369025], [0.9769022626482755, -0.20694513814567128, 0.007602143792902422, -1.116946285492455, 0.33671129595538396, 0.9348941188848842, -0.9742362659018642, 0.3841855024216996, -0.5492529925296089, 1.0872433194559101, 1.0224074110353782, -2.4502804893865626, 0.4741224606665683, 1.0586367880855143, -0.578971825502438, -0.22136518586369025], [0.9769022626482755, -0.20694513814567128, 0.007602143792902422, -1.116946285492455, 0.33671129595538396, 0.9348941188848842, -0.9742362659018642, 0.3841855024216996, -0.5492529925296089, 1.0872433194559101, 1.0224074110353782, -2.4502804893865626, 0.4741224606665683, 1.0586367880855143, -0.578971825502438, -0.22136518586369025]], [[-0.15436727838762757, -1.187741910139558, -0.11791444842706794, 1.6200151078549272, 0.4992446552165796, 1.586752076795199, 1.035065737434756, 1.2518367668529706, 0.13091240011652536, 0.8099439569844279, 2.1002217943852965, 1.534742697196371, 0.34011868915030824, -0.7217455935489808, -0.9549883153627023, 0.2537528581669864], [-0.15436727838762757, -1.187741910139558, -0.11791444842706794, 1.6200151078549272, 0.4992446552165796, 1.586752076795199, 1.035065737434756, 1.2518367668529706, 0.13091240011652536, 0.8099439569844279, 2.1002217943852965, 1.534742697196371, 0.34011868915030824, -0.7217455935489808, -0.9549883153627023, 0.2537528581669864], [-0.15436727838762757, -1.187741910139558, -0.11791444842706794, 1.6200151078549272, 0.4992446552165796, 1.586752076795199, 1.035065737434756, 1.2518367668529706, 0.13091240011652536, 0.8099439569844279, 2.1002217943852965, 1.534742697196371, 0.34011868915030824, -0.7217455935489808, -0.9549883153627023, 0.2537528581669864], [-0.15436727838762757, -1.187741910139558, -0.11791444842706794, 1.6200151078549272, 0.4992446552165796, 1.586752076795199, 1.035065737434756, 1.2518367668529706, 0.13091240011652536, 0.8099439569844279, 2.1002217943852965, 1.534742697196371, 0.34011868915030824, -0.7217455935489808, -0.9549883153627023, 0.2537528581669864], [0.46012695162139583, -1.9045817342802513, 0.23222421906122503, 0.36513287412683576, 0.318409458566481, -0.5668580127420708, 2.398263544594866, 0.9107897718504723, -1.4228340147390692, -0.7405895903642278, 0.8339025111981764, 0.5329341183962628, 0.13188270642753452, -0.03434879399637327, -0.22626784101888814, -0.47137749319831695], [0.46012695162139583, -1.9045817342802513, 0.23222421906122503, 0.36513287412683576, 0.318409458566481, -0.5668580127420708, 2.398263544594866, 0.9107897718504723, -1.4228340147390692, -0.7405895903642278, 0.8339025111981764, 0.5329341183962628, 0.13188270642753452, -0.03434879399637327, -0.22626784101888814, -0.47137749319831695], [0.46012695162139583, -1.9045817342802513, 0.23222421906122503, 0.36513287412683576, 0.318409458566481, -0.5668580127420708, 2.398263544594866, 0.9107897718504723, -1.4228340147390692, -0.7405895903642278, 0.8339025111981764, 0.5329341183962628, 0.13188270642753452, -0.03434879399637327, -0.22626784101888814, -0.47137749319831695], [0.46012695162139583, -1.9045817342802513, 0.23222421906122503, 0.36513287412683576, 0.318409458566481, -0.5668580127420708, 2.398263544594866, 0.9107897718504723, -1.4228340147390692, -0.7405895903642278, 0.8339025111981764, 0.5329341183962628, 0.13188270642753452, -0.03434879399637327, -0.22626784101888814, -0.47137749319831695]], [[-0.38543172339740256, -0.9007200768310993, -0.5933977196363743, 0.2745848303468262, -0.27456478787864735, -0.37645760091468844, 0.03752151346357176, 0.8328911117182467, -0.6982196221633257, 0.27763316225215845, 0.6409246854974151, -0.4603913104781554, -0.17083783404444233, -0.02386354485295921, -0.11672351355087592, 0.1329777346229887], [-0.30563553353689876, -0.22059179752494704, 0.9772044699427658, 1.139129606380705, 0.20141338037757267, -0.1802027849708767, -1.0742092881611218, 0.006657581555363106, -0.23944063885798927, -0.38860844331439737, 1.0816195103007553, -1.291055739354577, -0.09436979221103532, 0.2844182213413841, 0.12371098605592688, 1.0548961457089003], [-0.17536864673509156, -0.886824901111116, -0.07303109862415243, 0.6993176547577457, 0.024452967172760857, 0.11019803679673168, -0.46585231594894017, 0.8259138199949442, -0.581326491716226, 0.6679938475160673, 0.45795238933250937, -0.7411348211683604, -0.39016210404283524, -0.03798200250844902, -0.15060180357836617, 0.12700904724171197], [-0.3459573337420231, -0.08821892516821153, -0.4474791618320234, 0.04972672403378956, 0.3387960273755535, -0.7393613819849666, -0.3608835077286061, 0.28524648176558015, -0.0615548339399, -0.41903119300271324, 0.17099317119224558, 0.04854562893956761, 0.4364751556987254, 0.3225935294531564, -0.35643873699189343, 0.2269164742111283], [-0.5058772109771041, 0.06402591129742852, -0.5770381956885386, -0.7655924609170057, -0.15667823320503402, -0.5055251050730831, 0.18363851613798574, -0.3101067592537196, 0.08566655837973074, 0.17260483359933432, 0.036301839177353704, 0.14380753916260097, -0.18680264570637123, 0.5175794983010482, 1.2993143048249574, 0.1674749723881035], [0.1533502673827692, -0.3186769531202183, -0.7892284448728172, 0.100017311074506, -0.17621565009854503, -0.9263316364687337, 0.7748621225965473, 0.47078511220341845, -0.456462701701239, -0.509520996349734, -0.19247531894043987, 0.11152703598115489, -0.6431954357425899, -0.452240156805009, -0.3932801045675681, -0.11961500138498248], [0.14070772972745907, -0.0728240379217804, -0.8195310603546162, -0.007439931992411684, -0.19483480568175118, -0.8867940233682188, 0.7579815915778397, 0.263095667603214, -0.2851907879911578, -0.2655100708000702, -0.23272946411520148, 0.4432596570651372, -0.5942053725052645, -0.4359102472358996, 0.04617027523896295, -0.08931986215804587], [0.20277425723291242, 0.6774041799062801, 0.14044477443911157, -0.5697964282423941, -0.5160004675558527, -0.7922545165709612, 1.1462888245892433, -1.022465996011372, 0.1519935563897557, -0.653839726111031, -0.9359887402285378, 0.07273342447582216, 0.14340275822317622, -0.19661444840685713, 1.403715822112049, -0.7424519699774283]], [[-1.150634259919524, -0.19922423128611763, -0.4815707614477381, 0.1630720084024239, 0.5385674002611659, -0.6106684100417372, 0.27347214419289584, 0.06975618526302887, -0.5915415236404638, 1.0169141929695142, 0.3747575056310237, -0.1740558255058413, -0.3744554073266896, -0.07924053167024304, -0.02544768750902456, -0.37828357636281984], [-0.7231871360997535, -0.5199601212690653, -0.04054863733239661, 0.3645632923269953, 0.036408157815060585, 0.2882139369813621, 1.4073388847724033, 1.0075472899383109, 0.14563367317134338, 0.7914346477989465, 0.6006503937611058, 0.23602536321760031, 0.6203903828935897, 0.4685720486407464, -0.01833837998774233, 0.16856976448719038], [-0.8093263228356776, -0.5773943976065503, -0.21723452480074554, -0.0318734647621324, 0.34558208597144935, -0.22005869081239676, 0.9650060800313321, 0.30487924033986163, 0.2275915528364926, 0.5810489233965549, 0.628408501288642, -0.09073533363806455, 0.33012976159669677, 0.42023995852732815, -0.5845270943052614, -0.10265627192465279], [-0.9815332072037526, -1.0617715803257863, -0.07698349120478917, 0.1325080888941736, 0.31812664196306667, -0.2147101762487401, 1.2715077093180525, 0.594883786377156, 0.6050314106513757, 0.3566482142498991, 0.9305223644305323, 0.08602580920824228, 0.3001622766965844, 0.6001390184866006, -0.7083965088321905, -0.4473946538788633], [1.3120441871668553, -1.0257580237515476, 0.027162562640475427, 0.4421187999951296, 1.2577993159174774, -0.4798939833473116, -0.0558755410472586, -0.8552378796260949, -0.35899476361935334, 0.3466844495216249, 0.023581898353715595, 0.18385136017729412, 0.3685048566946401, 0.4679656720589321, 0.5262968858794655, -0.04091025998735467], [-0.1631615456312689, 0.24600917494778982, 0.7970672865451808, -0.3413349722939226, -0.2025499962128985, -1.1283554412398238, -0.474664057673736, -0.20316088620009104, -0.2556528628410016, 0.05001369608730143, -0.5617975195384324, 0.33202918346003973, -0.15823432872355672, -0.1929671090584752, 0.865045997987573, -0.054060262307430856], [-0.26966615273478667, 0.913912688662631, 0.6876544117481357, -0.02054688670468422, -0.08860453814370514, -1.8418395324302874, -0.6872318674557472, -0.10901891740231572, 0.6457685857377233, -1.0236772972424624, -0.4334905588323425, 1.1885342712542617, 0.27394411216897124, 0.35180895305860654, 1.0012032293114663, -0.0694387054371279], [-0.22047210675276863, 0.8602484534020641, 0.7175624215269086, 0.08435465755738901, -0.1802709259929367, -1.9678321784124195, -0.4651672961560244, 0.087473746149824, 0.6927445035040118, -0.9209440207468371, -0.34244991213563053, 1.131228812039574, 0.07297894144201969, 0.2902883422774684, 0.9151494194922019, -0.17637750697681642]]], "dtype": "float64"}, {"__nd__": [[[[-0.14544487154245192, -0.7164522571081597, -1.28919322719092, -0.6830203324000345, -1.0910181622054855, -0.9495899613521656, 0.114886001535766, -0.16705388559636833, 0.4878608415208204, -0.25653410585141756, 1.9879044407503788, -0.7698174393859692, -1.0494574911793721, 1.0539101395445558, -0.7445954455878293, 0.11776109258766718], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[1.0053015668910137, -0.1142963269697127, -0.38148103401332834, 0.9936009708862976, 0.33223679954700147, -0.5661107688153971, -1.2590703332499047, -0.4079930775147451, -1.7380189293092558, -0.9224034990272185, -1.1545697150335459, 0.28946781577853076, 0.7758850590296091, -0.6944417419171223, -0.47600169088170596, -1.163570066502072], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]], [[[-1.4558716071986488, -0.4699931875923203, 0.564651281355481, 0.46586731042272617, 0.5901614337728481, -0.3059991891543134, -2.8606225989825966, 0.941873547171145, 1.3517080423461578, 0.9267686892688831, 0.9187685992060648, 1.1214500855429523, 2.769683287961346, 0.3209113093482113, 1.6048923457166533, 0.8318099269703514], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[0.6970982940252463, -0.14309223040428232, -0.3627842298347233, -1.1739196317090077, 0.604586444674966, -0.32904182427704876, -1.2511243572359672, -0.7932021320713456, -1.237353184050747, 1.657955572756555, -0.2588428600301334, 0.10751022536384264, -1.2631742271058926, -1.077707360000256, 1.0853383975503472, -0.2482160614062073], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]], [[[-0.4276300729183073, 1.3049469505616562, 2.570092738692354, -0.7399058822942611, -2.174502072773742, -0.4254398963296527, 0.3768451073305704, 1.0501313269503614, 1.0790155135012853, -0.7839513663149491, -0.7932353410041735, 1.2673009187104822, -1.0923877024313569, 1.0902313512146344, -0.2268879272034576, 0.7664035884711488], [0.7061647137637838, 0.7766524810610778, 0.8204266971936063, -0.9893909801613543, 0.29752110861128966, 0.013982411315735277, 0.30840802864763195, 1.2332483942406183, -0.2711082411279299, -0.7119704226209977, -0.728115476185683, -0.31081837778290006, 0.7526105507610801, -0.8148156257479607, 1.1592578951012953, 0.29045146553250784], [-0.22611092064531074, -1.3316509777862344, 0.24493318448510143, 0.6914693220924303, -1.7657657867263103, 1.1919595712856506, 1.647747014893282, 0.338502296909925, 0.4917808983658258, 2.4136949965880823, 1.2318906798511962, -0.8977571989333307, -0.01664518614742305, -0.15693106761722703, -0.15087679609514876, -2.3642678177611556], [0.7570048405627458, -0.6367361170630729, 0.16365146447771292, -0.13613934374199768, 0.8710953770180234, -0.29580869027763773, -0.7954280607901872, 0.5324224800592481, 1.4048856848402156, -0.3198106833867411, 1.3699480278262324, 0.2621831500067624, -0.5327294852810801, 1.5251578003581547, -0.818626335553522, 1.0140481991147188], [-0.4981859667883459, 0.4502186456421907, -1.234148305277474, -0.8375926212541703, 1.0650523006996653, 0.18811685191459512, 0.03829301646633128, -0.43980824203442026, 0.446480673727825, -1.3972692775978013, -1.1199352443263735, -0.157790365329733, -0.8342810311785638, -0.7005937683055597, -0.07150325804844848, -0.13752133194645022], [-0.222506700637484, 1.6652756190688167, 1.0110758002746831, 1.2297345185288913, -0.028162001365168308, -0.5046842878231101, -1.4538991774513217, -0.34813740474730365, -2.085955516806786, -0.5837413170812261, 0.764238935433696, 1.621530244280291, 0.9656069166841026, -0.8616004422818208, 0.49261852280236235, -0.7891640250574642], [0.8229737900880982, -0.11136118318513945, 0.8582001144570633, 2.572895572233041, 0.9225243033323701, 2.3324499453787046, 0.177442312981226, -0.32971772068132815, 1.3337043892100016, 1.724356555267187, -0.5550145945569381, 0.07663196559606328, -1.4345295287904392, -0.20004587761769155, 0.8883322285022602, -0.08638483740324072], [-0.569220157581707, -1.407239902207787, 0.36825449991781684, 0.797540663665427, 0.5965006827757677, 0.25531569894951217, -1.2740653493386656, 1.7618086407811855, -0.05653636459299152, -0.14479645001033087, 0.7329285500357282, -2.163186235204895, -0.0644672373325154, -0.13678960297985446, 0.3722831319448136, 1.3471598477745907], [0.2267708786188296, 1.903719540005617, 2.1684157123550465, 0.7132322831037413, 0.504002187870548, 0.24362088066698953, -0.2981776943387448, 2.1019305233624266, -1.2637991429270883, -0.09444450704511713, -1.9996431856170793, -1.3159613447991658, 0.4179761229367063, 0.4737956461830622, 0.34948936620293497, -0.33738650439041584]], [[0.23207021517525864, -1.1791464394934508, 1.3586400261294171, -1.2926229107419693, 0.25441917324395535, 0.13490564635114768, -1.0507245554685565, 0.8869794016080271, -1.415958011229726, -0.683484532933555, -0.028702202916173543, -0.41249769672963527, 0.46780667461364833, 0.9240548410021371, -0.29377376425342144, -0.5414659678720907], [-0.5169782292637785, -0.3655939450055468, 0.8096488946941695, -0.2175850065483592, 0.40763953366453465, -0.09115277603330242, -0.844465175412494, 0.8637197398257774, -1.0380412580682665, 0.006197796707115902, 1.4870238550758998, -1.0931070420832962, -0.05518852081126953, -1.5847810984165838, -0.5094162712223194, -1.4945996585123718], [-0.3615871788340143, -0.5382974610574749, 0.13197161380603412, 1.5843009073214365, 0.28298451661725926, 0.6581744527499446, -0.05070193949372962, -0.24452361546620502, -0.8339818243786973, -1.800856820932183, 0.06862036839919183, 1.6291235191475462, 0.4762998718707188, -0.5339889586362624, 1.6494061436322494, -0.4326235019616808], [-0.6188397120633242, -1.212694351698304, -0.962050935083037, -0.2463628695717934, 1.999020667820133, -0.20617621326392488, -1.0315163014000943, -0.605418629431135, -0.2228617607558514, -1.7117519241888182, 0.12662843610718544, 0.7006684855746799, 0.3208122410050434, 0.6007326507656642, -1.2895470431032947, -1.6985919026466894], [1.2855745952249331, 1.0943313362419658, -1.5402644573983837, 0.20638165811738485, 0.025527869974148333, 0.5448862967059741, 0.20858621210957912, -0.5021319045882552, 0.19525663814289484, 0.15704688166820915, -0.0486276310087608, -1.8524186305219243, -2.995201022440529, -0.7901887934234801, -1.6347715081124043, 0.25256204274779664], [0.7160967175077158, 0.2631154237292073, -0.3695278958798835, 0.07048084459204089, 1.167341045550715, -0.7288080336039454, 0.9222947397293393, 0.7076163309240973, 0.2616842740988124, 1.4519301576671035, -0.7178185832067028, 0.9425060224272724, -0.9289515855680833, -0.7670315765631621, -1.059711085029156, 0.44583159760427166], [-0.9397600370828224, 0.05215989851629757, -1.136711911629779, 1.066257585784627, 1.8361735112481172, 0.22607664232280375, 1.6433654157233566, 0.5606636371484345, 2.169309982212192, 0.09971426544292412, -1.5704236041028392, 1.49058403082891, -0.12636714234675797, -1.0358450164480129, -0.6562658594674428, -1.6263317217708266], [-0.16770180360718803, -1.0712637551480848, 1.484180556178205, 0.6644919985471294, -1.6604060196052313, -1.5093723248953304, -0.5669757804700066, -1.9250410205953503, -0.10980580857164887, -0.04344814976476555, -0.7728063339237182, -0.5464943802106043, 0.498852844163285, 1.4216846227430395, 1.4956732179240155, 0.03430883063776426], [-0.7354851140141753, -0.5605398965196526, -0.39853710833038963, 1.1365535742141335, 1.5150308036745619, 0.45937210573959025, -0.5600021510349511, -0.973885944987591, 0.3160481273078601, -0.23779689710483434, -1.8960106100940826, 1.424341360634964, -0.008583035425387036, -0.3314610784330754, -0.9046298820827241, -0.9086011229733484]]], [[[1.4038291578515893, 0.5568982661709865, -1.3240953569990666, -2.201557117006762, 0.09988180022201963, 1.1820845467832928, 0.17374005455439948, 0.8820660345952854, -0.4846734103640801, -1.3639205623940929, 0.7485937326351536, 0.09894227464054638, 0.6836829270923552, -1.3815659418907666, 0.1815534413430963, -0.6255009899606692], [0.8065888431513611, -0.5806873242375763, -0.6073949019765076, -0.5857203770133427, 1.263717038467993, 0.028864195779211843, -0.7599971570164401, -1.0225841755527632, -1.0441224416609518, 1.4626275482563855, -1.814774513497246, -0.8503575320650164, 0.6378903416209587, 1.1784957119103587, -0.14827740739445017, 0.7855160537736622], [0.7596172911688887, 0.9861870768741756, 0.04641503077493213, 1.6549681110403118, 0.2161253725997985, 1.5482000856724596, -1.082788417359302, 0.96061305531659, -1.2350369202277605, -1.4755833724170708, 0.5254990763360979, -0.7997444816067071, 0.47699373504770903, -1.0392571047529968, 1.6403754882158161, 0.04541300736884309], [-1.0942737499358477, 0.21565910210810485, 0.9240799367594777, -0.7268860672016146, 0.5617496770275349, 2.093347711888077, 0.9154857742071912, 0.039247960186757934, -0.5195256010390428, 0.6291380372130817, -1.4644056944900845, -2.986478022301571, -0.613397013531575, -1.0284662209338726, 1.799839710587298, 0.19588654490386795], [0.9341038038726168, -0.9809394388777001, 0.09944872702975784, 0.6594175075155629, 1.916430660845645, 0.6280729155292678, -0.035543979190243234, -0.4572617685607399, 0.889372065743123, 1.0439059502274242, -0.44265502454534617, 0.6105468435269886, -0.14369760838662038, -1.686521242221457, -1.046413692527123, 0.8411470639005992], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]], [[-0.0909370888182374, -0.06102960599713808, -1.0791675863890429, -0.23909435282450797, 0.28724736695517095, 0.5837607504782176, 1.295664503057182, 0.057048197180256645, -0.4357512542885165, 0.427944847082082, -0.6054584854017475, -1.2284018657621247, 1.1658174133660713, -0.35966252750539546, 0.14188858466737753, 0.6701044983880885], [-1.087502107535094, 0.6610705932032451, 0.47000781019778404, -0.0786951965212257, 0.413828570903945, -0.7375753310221359, -2.186627825908461, -0.26442112461867023, -0.8926158968589242, 0.8870077749338153, 0.282840295701735, -0.511611700526905, 0.0655950955625179, 1.3710853715756686, 1.1530663941817965, 0.386902226933139], [0.4485263830803767, -0.2604080417729926, -0.11286324321516372, 0.6066693824226794, -0.10218464932996439, 0.31613473140693815, -1.098923752533973, -1.652323167168886, -1.6826912560069485, -0.42126579609714815, 1.3442906422568701, 1.1995958076434512, -0.9814296623325831, -1.5408069593316496, 0.8088977952524207, 0.47898577886724764], [-1.3532677622094842, 1.0735628380399151, 0.19024643307954248, 1.3394518668961404, 0.3579726340873901, 0.9059391704119965, 0.3368444827337588, -0.9376466937313434, -0.2178904256607671, -0.6630188479906014, 0.8452523143858375, -0.9817645515074065, -1.51943281101853, 0.6968105828302908, 1.1609312676187333, 1.4599091358239493], [1.4986090579948839, 0.7053757558894472, -0.21011368424475652, -0.7444566477548312, 0.5577568332992092, 0.34395172820801173, -1.681218971972077, 0.6770060021095777, -0.7349183462739384, 0.698344279432918, 0.558303354154615, 0.7546725929699328, 1.1140404809054816, 0.7328285620708094, -1.0901414224087804, -1.0578753954286788], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0], [40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0810468379366278, 0.49670085743436004, -1.6323219323519624, -2.084963671876908, 0.9300367521567386, 2.3133719216161226, 0.8620755189271605, -0.24423687921083367, 0.28050634678012704, -2.828253250623994, -1.0346619100677745, 1.0564668867073979, 0.898156332357861, -2.207805125910652, 1.232281008101371, 0.5591688855045999], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[0.9769022626482755, -0.20694513814567128, 0.007602143792902422, -1.116946285492455, 0.33671129595538396, 0.9348941188848842, -0.9742362659018642, 0.3841855024216996, -0.5492529925296089, 1.0872433194559101, 1.0224074110353782, -2.4502804893865626, 0.4741224606665683, 1.0586367880855143, -0.578971825502438, -0.22136518586369025], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]], [[[-0.15436727838762757, -1.187741910139558, -0.11791444842706794, 1.6200151078549272, 0.4992446552165796, 1.586752076795199, 1.035065737434756, 1.2518367668529706, 0.13091240011652536, 0.8099439569844279, 2.1002217943852965, 1.534742697196371, 0.34011868915030824, -0.7217455935489808, -0.9549883153627023, 0.2537528581669864], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[0.46012695162139583, -1.9045817342802513, 0.23222421906122503, 0.36513287412683576, 0.318409458566481, -0.5668580127420708, 2.398263544594866, 0.9107897718504723, -1.4228340147390692, -0.7405895903642278, 0.8339025111981764, 0.5329341183962628, 0.13188270642753452, -0.03434879399637327, -0.22626784101888814, -0.47137749319831695], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]], [[[0.45348924855482103, 0.3490698320618196, -0.14843380445321855, 1.241103706461563, 0.02043960560289747, -1.177624501666097, 0.2978175835368062, 0.12318903722818446, -0.040502829002204206, 0.23232640681619574, 2.3495531721956855, 0.23351901997783128, 0.457794753077021, -0.24645480856286245, 1.3103865519597715, -0.05656058864668221], [-0.8021083355278161, -1.2226428849539965, 0.5270918831485563, 0.7442358784331412, -1.588551827774419, -0.7237141069722046, 0.6413868499110363, 1.0271160617787196, -1.3212260599450445, -0.4087118552942355, 0.831768869291634, -0.5348942725265395, -0.4517353828944098, -0.5417974689822959, -1.3082161382774413, 0.04540970767256627], [0.3029250715030712, -1.4035863277668814, 0.09848556594080901, 1.1060581160781187, 0.20567148048059108, 0.9453840155840322, -0.5513148318244018, 1.3672061393525128, -0.7881364875171292, 1.8833833034706247, -0.4511039082529584, -0.7207785056850035, -0.8949810818089227, -0.41504493138051307, -0.4955772466526089, -0.6917725638406813], [-0.33178500902044955, 0.6620391064287441, 0.8797226429138996, -0.3400291332547872, 1.2321465640163338, -1.000674009730229, -1.0312794134437753, -1.1338053955310952, 0.5986278853586913, -1.5538412220643407, -1.399834668218403, 1.2750935383100084, 1.7088872697564919, 1.3391127644647582, -1.4963747550383706, 0.0065392554304298115], [0.826159701872232, 0.09072638217942289, -1.6242270588344543, -0.3487217781640566, -1.7051393371808987, -1.2394411568718704, 2.1451520357429232, 0.6719362710029061, -1.4750693303909352, -1.5086182003094526, 0.4481253905037407, -0.6179530470677405, 0.5686865679295522, -0.6563408232622712, -0.20918554944812545, 0.22913804836699356], [-1.0425843086187383, -1.930194037646372, -1.204502626035393, -0.7088424652129783, -0.21507838266500906, -0.0706887691553342, -0.07921269392390515, 0.5019532202178966, -0.9050357596317037, 1.382136313697382, 1.988347489154716, 0.3520614266417755, 0.1739412726734303, 1.2332775946474066, 0.6337528708044734, 0.17293903103560176], [-0.18279434902449157, -0.7540787671055834, -1.1843869781421177, 0.6499515036791333, 1.17223424297214, -1.0725984514356413, -0.08787476349616176, 0.2021257572271039, -0.44147812189575236, -0.7774019533764328, -0.40872577854715136, -1.986573793588022, -0.07726774500169377, -0.3106481701334202, 2.1002647416741365, 0.9197710972652077], [-0.36233625364362343, -0.02498371325575958, 1.6989366631605227, 1.588795013006287, 0.16741353315593088, -0.1457453698422363, -1.46231693200154, -0.34448553044702235, -0.18632164207772312, -0.7531993282122036, 1.7125871847761867, -1.8930187151158349, -0.15312954802751536, 0.3895240579324037, 0.419251660827471, 1.6691079751612958], [-1.072822420682492, 0.5879554915597687, -2.898614617024146, -0.774015279292408, 1.2674952300868871, -0.8363261869699357, -1.2632614362953578, 1.9392464493678783, 1.0507760296067028, -0.18549136253838838, -0.6394923529564971, -0.27503010905511827, -0.39684249843042757, -0.2885423471166199, -0.804998969017241, 0.5216492324305687]], [[-0.43407506437830745, 1.252468159533808, 0.2040317137638252, -0.7745705329496348, -0.797057565340283, -1.195201052013818, 1.1866185535233382, -2.040504380600246, 0.25173079304344975, -1.4622981361005167, -1.0827615837006996, -1.512913408289115, 0.9379880811278674, -0.10783298845229917, 2.7162272789621746, -1.064976433074287], [-1.1233488174763162, 0.5610081682206371, 0.5322913211026215, -0.312752579841828, -2.052834922739572, 1.4116051772766451, 0.5343497679978567, 0.36350691367384036, -0.6297993974836996, 0.603452291181055, -0.5992394254479873, 0.002782362028699419, -0.23031125033700295, 1.215344509742451, -0.5142291775100767, -0.39040138695912274], [1.0593440260021498, 0.17986184518242124, -0.06755071567350249, -0.2900110919043919, 0.9010508614757021, -0.7011904877438835, 0.4197467627090478, 1.9785337005133665, 0.34653185158580296, -0.8393389532624606, -1.2570782775538027, 0.0913928921761727, -1.2842551323548932, -1.6518002934310132, -0.8178593240212917, -1.1548838798059364], [0.8153216554309594, -1.5129322089603954, 1.5206858984910974, -0.8151857651342108, 0.054662978872448975, 0.6076080292332172, 0.15800535486330722, 0.6552466645712847, -1.054365703737103, -2.341079099388513, -0.270742250497455, -1.7528945955959814, -0.6525186158748978, 0.8720540969383103, -0.27137987374942657, -0.6681801860042257], [-1.205816597844541, -0.36867168433691105, -0.25081680293783243, -1.6610216668350122, 0.21573514740291905, -1.1555926876966116, 0.13841683858152587, -0.532583247535998, 0.9094871454462922, 0.775747641036947, -0.07191889499725981, -0.18943499544916995, 0.35863452374958005, 1.3057222449014558, 0.9909160243519445, 0.08859570606956685], [0.24926498110087889, 1.5156350108956544, -1.6955165561987913, 0.4848982329631039, -0.9567307938507938, 1.3039183586818974, -0.5436124326602008, -0.15811483591241962, -1.13066503564093, -0.024170144096250457, 0.7569795471893374, 1.6395380071343677, -1.2954630823226267, -0.4699377462582708, 3.8864552710410796, 0.9217358897855625], [-0.025641660071995545, 0.5070818598060605, -2.681901571333781, 1.60351971804367, 0.14751459445916568, -1.4456269739712746, 1.1621213844890788, -0.6370449948071236, -1.0342037330804474, 0.5195441639314857, 0.3987098615681088, -0.46217542782828724, -0.07581267564668939, -1.3466926889404711, -1.4483717961252065, 0.8991607563497701], [1.3172838676535006, 0.4312852345935342, 0.37363965609735533, -0.3960975965738327, -0.13019312759828652, -0.9373248956610112, 1.758270921248861, -0.5680427060002515, 0.41060240157624794, 0.00016674677896742597, -1.4002809519865258, 1.9391935035563979, -0.3758299307032596, -0.7687258613081643, 0.11466400875383675, -0.8982965864809159], [-0.6191295508246943, -1.9232642651625773, -1.3860513922345854, -0.04642735433419728, -1.1060513779491727, -2.6173164194237817, 1.5012597319609273, 0.4586703067977707, -0.5503271136299331, -0.6016890085218324, 0.46896230646529885, 1.3587476828431804, -0.7107702239060719, 0.3593028784122634, -0.7150284735470539, 0.2684755280813338]]], [[[-0.05049641095083523, 0.5057614826048096, -1.1841302440536672, -2.098577562735211, -0.00016556387578946797, -0.4367785013228669, 0.09812105876600097, 0.6559041690469416, -0.2033843250993628, 1.2112682825036059, -0.024050770185300285, 0.13227414545791594, 0.855094960218829, 0.6115973107692956, -0.06459700496052836, 0.15147794717976315], [-0.7325077075725053, -0.5885402327147731, 0.13132265828990042, 0.8462540265376409, -0.4059438232776296, 0.8735382871450506, 1.9794363955328722, 2.1877755810645407, 0.06357245403903584, 1.095914334286998, 0.652855865054215, 0.8306879240179876, 0.884154477405213, 0.570092253775361, 0.875439245116228, 0.24068192177543613], [-0.20984597803395438, 0.30575821533983477, 0.5470753539463475, 1.28047588518512, 0.4263070672726552, 0.7989929010721081, 1.4833457972471682, -0.8798562848592036, -0.20736754043443523, 0.5968579591827916, 0.08149553187709808, -0.9605385504148742, 1.0465215571750213, -0.015999235517005012, -0.9484673608574571, 1.9530872597100337], [-1.232303911515722, 0.03862752955704673, -0.6207221489901279, 0.16117169074175516, 0.6161871156697357, -0.7686342067399412, -0.05466770264738846, -0.08203095697633514, -0.9570086194403548, 1.216826196312125, 0.22016793539333407, -0.24215134445472328, -0.6210333963155036, -0.28904388032810574, 0.18836057398498784, -0.4175319404494895], [-1.2490682262713597, -1.9662644726986416, 0.017614937041686207, -0.1395605118820667, 0.6751620988202943, -0.7658265377654894, 1.3478297548091558, 0.024913568219149716, 1.524070517801729, -0.42700390998073806, 1.4880909584078001, -0.14079386165808747, 0.0726676372661932, 0.9252985108569339, -1.9058651257910082, -1.1777333935736278], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]], [[0.8960789585932066, -0.40792916481990404, 0.9048473795644835, 0.5957071916576799, 1.4620180665716653, 0.06936802967455195, -0.14207718027474814, -0.4148282000124647, -0.2837981802135376, -0.6733856307956471, -0.24865063620055392, 0.5934916895880746, 0.5493718052575148, 1.4085243644565233, 1.1823715292118242, 0.7123072141644236], [-1.0688020852532705, 0.6528002570395969, 1.2463039776292244, -1.225101651231791, -1.0397617546110227, -0.8794495187402187, -0.9670960473664579, -0.21878888911238828, -0.8675790226551632, 0.45446221707678186, -1.1485436386065973, -0.06447743418482642, -0.4381052537646047, -0.909855889510612, 1.0947980204499768, 0.08675415760012148], [0.6002408553014308, -0.27574187646611276, 0.7611515544215757, 0.6721111673832539, -0.09537194545221389, -1.717910238177568, 0.8777901325375698, 0.7497206522219166, 0.23805901725218487, 0.2999132827833922, 0.18103458729481264, 0.2500183974042268, -0.9478056260663629, 0.001718988139928557, 0.37575238954691337, -0.5547694668570979], [-0.9129834258945715, 1.9173918762969722, 0.6651930159072552, -0.24874803952609156, -0.434376402431708, -2.5850457856618068, -1.3675827684394424, -0.19130937784917285, 1.3467440200733614, -1.9998259095826407, -0.6417015069465, 1.9730949242733973, 0.7285887918640518, 0.473569638666792, 1.2539915649475473, -0.06093714534384334], [1.826289676185761, -1.634458097876963, -0.5896575671954235, 0.4900469922667428, 1.4412212133691968, -0.5967857202585577, 0.06297177296515619, -1.241494570113807, -0.46837959642252563, 0.9917968560582078, 0.26963185086493774, -0.10051772812703341, 0.3737700871414614, 0.07718806994273542, 0.11133663404623863, -0.42843394393473433], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0], [-30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [1, 1, 9, 5], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[-1.9156455579583005, 1.1018018558224842], [-0.3299040738738497, -0.8806465179277969]]], "dtype": "float64"}, {"__nd__": [[[-0.6562850109290311, -0.6720146806586559]]], "dtype": "float64"}, {"__nd__": [[[0.3801890922489917, -0.11005869527805158]]], "dtype": "float64"}, {"__nd__": [[[[-0.6517911526116896, -0.17471729232577715], [1.6637239913911968, 0.659147749832255], [-1.6413972945846467, -0.005203264171931977], [40.0, 40.0]]]], "dtype": "float64"}, {"__nd__": [[[[-1.608187784186389, 0.2417718768768513], [0.23538091873745476, 1.5756260314314627], [0.3166450164719021, 0.5105466616976417], [-30.0, -30.0]]]], "dtype": "float64"}, {"__nd__": [3], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[0.008441929277548273, 0.4119537847881304], [-0.16879644076505396, 0.31642381474304176]]], "dtype": "float64"}, {"__nd__": [[[[-0.6517911526116896, -0.17471729232577715], [1.6637239913911968, 0.659147749832255], [-1.6413972945846467, -0.005203264171931977], [-0.6562850109290311, -0.6720146806586559]]]], "dtype": "float64"}, {"__nd__": [[[[-1.608187784186389, 0.2417718768768513], [0.23538091873745476, 1.5756260314314627], [0.3166450164719021, 0.5105466616976417], [0.3801890922489917, -0.11005869527805158]]]], "dtype": "float64"}, {"__nd__": [4], "dtype": "int64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(decode_step, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")